# GBM Paper Follow-Up

## Load data and metadata

In [ ]:
# Autoreload extension for IPython/Jupyter
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('load_ext', 'autoreload')
        ipython.run_line_magic('autoreload', '2')
except (ImportError, NameError):
    pass

import pandas as pd
import numpy as np
import anndata as ad
import s3fs

import scanpy as sc
import matplotlib.pyplot as plt
import openslide
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter, AllDataCenters

sc.set_figure_params(dpi=200, color_map="viridis_r")
sc.settings.verbosity = 0

from cloudpathlib import S3Path
from scipy.sparse import issparse
from joblib import Parallel, delayed
from tqdm import tqdm

from gbmcollaboration.definitions import (
    DATA_PATH_CHUV_FILES,
    DATA_PATH_UKER_FILES,
    DATA_PATH_CR_FILES,
)
from gbmcollaboration import AllDataCenters

%load_ext rpy2.ipython

### Load clinical data

In [ ]:
import pandas as pd
from gbmcollaboration import CHUVDataCenter, UKERDataCenter, CRDataCenter

# Load processed clinical tables for all centers and combine into one cohort

def load_processed_clinical(dc, center_name):
    d = dc.load_tabular(exclude_idh1_mut=False)
    clin = d["clinical"]
    key = next((k for k in clin if "processed" in str(k).lower() and "clinical" in str(k).lower()), None)
    df = clin[key].copy() if key else list(clin.values())[0].copy()
    df["center"] = center_name
    return df

ch = load_processed_clinical(CHUVDataCenter, "CHUV")
uk = load_processed_clinical(UKERDataCenter, "UKER")
cr = load_processed_clinical(CRDataCenter, "Charite")

clin_all = pd.concat([ch, uk, cr], ignore_index=True, sort=False)
clin_out = clin_all.copy()

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

### Check Cox HR assumptions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from scipy.ndimage import uniform_filter1d
import warnings
warnings.filterwarnings("ignore")

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")

# =============================================================================
# CONFIG
# =============================================================================

TIME_COL  = "os_years"
EVENT_COL = "os_event"
MGMT_COL  = "MGMT promoter methylation"
AGE_COL   = "age_at_diagnosis"
SEX_COL   = "Administrative gender"

COL_METH   = "#ff7b02"
COL_UNMETH = "#007ebc"

SAVE_DIR = "/home/sagemaker-user/gbm-collaboration_2/analysis_survival"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# PREPARE DATA
# =============================================================================

clin_ph = clin_out.copy()
clin_ph["_priority"] = clin_ph["Sample collection chronology"].map(
    {"Baseline": 0, "At recurrence": 1}).fillna(2)
clin_ph = clin_ph.sort_values(["Subject Id", "_priority"])
clin_ph = clin_ph.drop_duplicates(subset=["Subject Id"], keep="first")
clin_ph = clin_ph.drop(columns=["_priority"])

clin_ph["time"]  = pd.to_numeric(clin_ph[TIME_COL],  errors="coerce")
clin_ph["event"] = pd.to_numeric(clin_ph[EVENT_COL], errors="coerce")
clin_ph["mgmt"]  = (clin_ph[MGMT_COL].astype(str) == "Methylated").astype(int)
clin_ph["sex"]   = (clin_ph[SEX_COL].astype(str) == "Female").astype(int)
age_vals         = pd.to_numeric(clin_ph[AGE_COL], errors="coerce")
clin_ph["age_z"] = (age_vals - age_vals.mean()) / age_vals.std()

df = clin_ph[["time", "event", "mgmt", "age_z", "sex"]].dropna()
df = df[df["time"] > 0].copy()

n_meth   = int(df["mgmt"].sum())
n_unmeth = int((df["mgmt"] == 0).sum())
median_os = float(df[df["event"] == 1]["time"].median())

print(f"N = {len(df)},  events = {int(df['event'].sum())}")
print(f"MGMT methylated: {n_meth} / {len(df)}")
print(f"Median OS (events): {median_os:.2f} years")

with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv["surv_data"] = ro.conversion.py2rpy(df)

# =============================================================================
# COX + SCHOENFELD (already run, re-run cleanly here)
# =============================================================================

ro.r("""
cox_fit  <- coxph(Surv(time, event) ~ mgmt + age_z + sex,
                  data = surv_data, ties = "efron")
zph_test <- cox.zph(cox_fit)
zph_id   <- cox.zph(cox_fit, transform = "identity")

# KM curves
km_meth   <- survfit(Surv(time, event) ~ 1,
                     data = surv_data[surv_data$mgmt == 1,])
km_unmeth <- survfit(Surv(time, event) ~ 1,
                     data = surv_data[surv_data$mgmt == 0,])
km_all    <- survfit(Surv(time, event) ~ 1, data = surv_data)

t_meth   <- km_meth$time;   s_meth   <- km_meth$surv
t_unmeth <- km_unmeth$time; s_unmeth <- km_unmeth$surv
t_all    <- km_all$time
h_all    <- km_all$n.event / (km_all$n.risk *
             c(diff(c(0, km_all$time))))

# Schoenfeld residuals
scho_t    <- zph_id$time
scho_mgmt <- zph_id$y[, "mgmt"]
scho_age  <- zph_id$y[, "age_z"]
beta_mgmt <- zph_id$y[, "mgmt"]

lo_mgmt  <- lowess(scho_t, scho_mgmt, f = 0.5)
lo_age   <- lowess(scho_t, scho_age,  f = 0.5)
lo_beta  <- lowess(scho_t, beta_mgmt, f = 0.5)

ref_beta_mgmt <- coef(cox_fit)["mgmt"]
ref_beta_age  <- coef(cox_fit)["age_z"]

# O/E ratio — compute manually per group
# Methylated group
km_log_meth   <- survfit(Surv(time, event) ~ 1,
                          data = surv_data[surv_data$mgmt == 1,])
km_log_unmeth <- survfit(Surv(time, event) ~ 1,
                          data = surv_data[surv_data$mgmt == 0,])

# Cumulative observed events
cum_obs_meth   <- cumsum(km_log_meth$n.event)
cum_obs_unmeth <- cumsum(km_log_unmeth$n.event)

# Expected under PH: scale by overall event rate
total_events <- sum(surv_data$event)
total_n      <- nrow(surv_data)
prop_meth    <- sum(surv_data$mgmt) / total_n
prop_unmeth  <- 1 - prop_meth

# Cumulative expected (simple proportional)
cum_exp_meth   <- (1:length(cum_obs_meth))   * prop_meth   * 0 +
                   cumsum(km_log_meth$n.event   / km_log_meth$n.risk   *
                          km_log_meth$n.risk)
cum_exp_unmeth <- cumsum(km_log_unmeth$n.event / km_log_unmeth$n.risk *
                          km_log_unmeth$n.risk)

# Use log-rank expected counts approach via survdiff
sd_mgmt <- survdiff(Surv(time, event) ~ mgmt, data = surv_data)
""")

mgmt_zph_p   = float(ro.r("zph_test$table['mgmt',   'p']")[0])
age_zph_p    = float(ro.r("zph_test$table['age_z',  'p']")[0])
sex_zph_p    = float(ro.r("zph_test$table['sex',    'p']")[0])
global_zph_p = float(ro.r("zph_test$table['GLOBAL', 'p']")[0])

t_meth   = np.array(ro.r("t_meth"));   s_meth   = np.array(ro.r("s_meth"))
t_unmeth = np.array(ro.r("t_unmeth")); s_unmeth = np.array(ro.r("s_unmeth"))
t_all    = np.array(ro.r("t_all"));    h_all    = np.array(ro.r("h_all"))
scho_t   = np.array(ro.r("scho_t"))
scho_mgmt = np.array(ro.r("scho_mgmt"))
scho_age  = np.array(ro.r("scho_age"))
beta_mgmt = np.array(ro.r("beta_mgmt"))
lo_sx_m  = np.array(ro.r("lo_mgmt$x")); lo_sy_m = np.array(ro.r("lo_mgmt$y"))
lo_sx_a  = np.array(ro.r("lo_age$x"));  lo_sy_a = np.array(ro.r("lo_age$y"))
lo_tx    = np.array(ro.r("lo_beta$x")); lo_ty   = np.array(ro.r("lo_beta$y"))
ref_hr_mgmt = float(np.exp(ro.r("ref_beta_mgmt")[0]))
ref_hr_age  = float(np.exp(ro.r("ref_beta_age")[0]))

# O/E via survdiff — extract observed and expected
ro.r("""
oe_obs_vals <- sd_mgmt$obs
oe_exp_vals <- sd_mgmt$exp
oe_labels   <- names(sd_mgmt$obs)
""")
oe_obs_vals = np.array(ro.r("oe_obs_vals"))
oe_exp_vals = np.array(ro.r("oe_exp_vals"))

print(f"\nOverall O/E (MGMT):")
print(f"  Methylated:   O={oe_obs_vals[1]:.0f}, E={oe_exp_vals[1]:.1f}, O/E={oe_obs_vals[1]/oe_exp_vals[1]:.2f}")
print(f"  Unmethylated: O={oe_obs_vals[0]:.0f}, E={oe_exp_vals[0]:.1f}, O/E={oe_obs_vals[0]/oe_exp_vals[0]:.2f}")

# For time-varying O/E, compute from KM directly
ro.r("""
# Compute time-varying OE using counting process approach
# Split follow-up into intervals and compute O/E per interval
breaks  <- quantile(surv_data$time[surv_data$event==1],
                    probs = seq(0, 1, by = 0.1))
breaks  <- unique(c(0, breaks))
n_int   <- length(breaks) - 1

oe_time  <- numeric(n_int)
oe_ratio_meth   <- numeric(n_int)
oe_ratio_unmeth <- numeric(n_int)

for (i in 1:n_int) {
    t_lo <- breaks[i]
    t_hi <- breaks[i + 1]
    oe_time[i] <- (t_lo + t_hi) / 2

    sub <- surv_data[surv_data$time >= t_lo,]
    sub$time2  <- pmin(sub$time, t_hi)
    sub$event2 <- sub$event * (sub$time <= t_hi)

    if (sum(sub$event2) < 2) next

    sd_int <- tryCatch(
        survdiff(Surv(time2, event2) ~ mgmt, data = sub),
        error = function(e) NULL
    )
    if (!is.null(sd_int) && all(sd_int$exp > 0)) {
        oe_ratio_meth[i]   <- sd_int$obs[2] / sd_int$exp[2]
        oe_ratio_unmeth[i] <- sd_int$obs[1] / sd_int$exp[1]
    } else {
        oe_ratio_meth[i]   <- NA
        oe_ratio_unmeth[i] <- NA
    }
}
""")

oe_time         = np.array(ro.r("oe_time"))
oe_ratio_meth   = np.array(ro.r("oe_ratio_meth"), dtype=float)
oe_ratio_unmeth = np.array(ro.r("oe_ratio_unmeth"), dtype=float)

# Replace 0 with NaN (unfilled intervals)
oe_ratio_meth[oe_ratio_meth == 0]     = np.nan
oe_ratio_unmeth[oe_ratio_unmeth == 0] = np.nan

# =============================================================================
# FIGURE 1 — PH DIAGNOSTICS
# =============================================================================

fig1 = plt.figure(figsize=(14, 11))
gs   = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)
ax1  = fig1.add_subplot(gs[0, 0])
ax2  = fig1.add_subplot(gs[0, 1])
ax3  = fig1.add_subplot(gs[1, 0])
ax4  = fig1.add_subplot(gs[1, 1])

# Panel 1: Log-log plot
mask_m = (s_meth   > 0) & (s_meth   < 1) & (t_meth   > 0)
mask_u = (s_unmeth > 0) & (s_unmeth < 1) & (t_unmeth > 0)
ax1.plot(np.log(t_meth[mask_m]),   np.log(-np.log(s_meth[mask_m])),
         color=COL_METH,   linewidth=1.5, label=f"Meth. (n={n_meth})")
ax1.plot(np.log(t_unmeth[mask_u]), np.log(-np.log(s_unmeth[mask_u])),
         color=COL_UNMETH, linewidth=1.5, label=f"Unmeth. (n={n_unmeth})")
ax1.set_xlabel("log(time)", fontsize=10)
ax1.set_ylabel("log(−log(S(t)))", fontsize=10)
ax1.set_title("Log-log survival plot (MGMT)\nParallel lines → PH satisfied",
              fontsize=10, fontweight="bold")
ax1.legend(fontsize=9, frameon=False)
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Panel 2: Scaled Schoenfeld residuals — show both MGMT and Age
ax2.scatter(scho_t, scho_age, color="gray", alpha=0.35, s=12, zorder=2,
            label="Age residuals")
ax2.plot(lo_sx_a, lo_sy_a, color="#C0392B", linewidth=2.0, zorder=3,
         label=f"Age smooth (p={age_zph_p:.3f})")
ax2.scatter(scho_t, scho_mgmt, color="lightblue", alpha=0.35, s=12, zorder=2)
ax2.plot(lo_sx_m, lo_sy_m, color=COL_UNMETH, linewidth=2.0, zorder=3,
         linestyle="--", label=f"MGMT smooth (p={mgmt_zph_p:.3f})")
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.7)
ax2.set_xlabel("Time (years)", fontsize=10)
ax2.set_ylabel("Scaled Schoenfeld residual", fontsize=10)
ax2.set_title("Scaled Schoenfeld residuals\nTime trend → PH violated",
              fontsize=10, fontweight="bold")
ax2.legend(fontsize=8, frameon=False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# Panel 3: Time-varying HR (MGMT)
ax3.scatter(scho_t, np.exp(beta_mgmt), color="gray", alpha=0.4, s=15, zorder=2)
ax3.plot(lo_tx, np.exp(lo_ty), color="#C0392B", linewidth=2.0, zorder=3,
         label="Time-varying HR (lowess)")
ax3.axhline(ref_hr_mgmt, color="#2C3E50", linewidth=1.2, linestyle="--",
            label=f"Cox HR = {ref_hr_mgmt:.2f} (constant)")
ax3.axhline(1.0, color="black", linewidth=0.6, linestyle=":", alpha=0.5,
            label="HR = 1")
ax3.set_xlabel("Time (years)", fontsize=10)
ax3.set_ylabel("Hazard ratio (meth. vs unmeth.)", fontsize=10)
ax3.set_title("Time-varying hazard ratio (MGMT)\nFlat line → PH satisfied",
              fontsize=10, fontweight="bold")
ax3.legend(fontsize=9, frameon=False)
ax3.set_ylim(bottom=0)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)

# Panel 4: Time-varying O/E ratio
valid_m = ~np.isnan(oe_ratio_meth)
valid_u = ~np.isnan(oe_ratio_unmeth)
ax4.plot(oe_time[valid_m], oe_ratio_meth[valid_m],
         color=COL_METH,   linewidth=1.5, marker="o", markersize=4,
         label=f"Meth. (n={n_meth})")
ax4.plot(oe_time[valid_u], oe_ratio_unmeth[valid_u],
         color=COL_UNMETH, linewidth=1.5, marker="o", markersize=4,
         label=f"Unmeth. (n={n_unmeth})")
ax4.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.7,
            label="O/E = 1 (PH satisfied)")
ax4.set_xlabel("Time (years)", fontsize=10)
ax4.set_ylabel("O/E ratio (per interval)", fontsize=10)
ax4.set_title("Observed / Expected events over time\nConstant ratio → PH satisfied",
              fontsize=10, fontweight="bold")
ax4.legend(fontsize=9, frameon=False)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)

fig1.suptitle(
    f"Proportional hazards diagnostics — MOSAIC GBM cohort (N={len(df)})\n"
    f"Schoenfeld p: MGMT={mgmt_zph_p:.3f}  Age={age_zph_p:.3f}  "
    f"Sex={sex_zph_p:.3f}  Global={global_zph_p:.3f}",
    fontsize=10, fontweight="bold", y=1.01)

fig1.savefig(f"{SAVE_DIR}/ph_diagnostics_cohort.png",
             dpi=300, bbox_inches="tight")
fig1.savefig(f"{SAVE_DIR}/ph_diagnostics_cohort.pdf",
             bbox_inches="tight")
plt.show()
plt.close(fig1)
print("Saved ph_diagnostics_cohort.png/.pdf")

# =============================================================================
# FIGURE 2 — TAU SELECTION
# =============================================================================

h_smooth = uniform_filter1d(h_all, size=max(1, len(t_all) // 15))

fig2, (ax_km, ax_hz) = plt.subplots(1, 2, figsize=(12, 5))

for t_arr, s_arr, color, label in [
    (t_meth,   s_meth,   COL_METH,   f"MGMT meth. (n={n_meth})"),
    (t_unmeth, s_unmeth, COL_UNMETH, f"MGMT unmeth. (n={n_unmeth})"),
]:
    t_plot = np.concatenate([[0], t_arr])
    s_plot = np.concatenate([[1], s_arr])
    ax_km.step(t_plot, s_plot, where="post",
               color=color, linewidth=1.5, label=label)

ax_km.axvline(1.5, color="gray",  linestyle="--", linewidth=1.0, alpha=0.8)
ax_km.axvline(3.0, color="black", linestyle="--", linewidth=1.0, alpha=0.8)
ax_km.text(1.55, 0.97, "τ=1.5y", fontsize=8, color="gray",  va="top")
ax_km.text(3.05, 0.97, "τ=3y",   fontsize=8, color="black", va="top")
ax_km.set_xlabel("Time (years)", fontsize=10)
ax_km.set_ylabel("Survival probability", fontsize=10)
ax_km.set_title("Kaplan-Meier by MGMT status\nwith tau markers",
                fontsize=10, fontweight="bold")
ax_km.set_ylim(0, 1.05)
ax_km.set_xlim(0)
ax_km.legend(fontsize=9, frameon=False)
ax_km.spines["top"].set_visible(False)
ax_km.spines["right"].set_visible(False)

ax_hz.plot(t_all, h_smooth, color="#2C3E50", linewidth=2.0,
           label="Smoothed hazard (cohort)")
ax_hz.axvline(1.5, color="gray",  linestyle="--", linewidth=1.0, alpha=0.8,
              label=f"τ=1.5y (≈ median OS {median_os:.1f}y)")
ax_hz.axvline(3.0, color="black", linestyle="--", linewidth=1.0, alpha=0.8,
              label="τ=3y (inflection)")
ax_hz.set_xlabel("Time (years)", fontsize=10)
ax_hz.set_ylabel("Smoothed hazard rate", fontsize=10)
ax_hz.set_title("Population hazard rate over time\n"
                "Inflection and plateau justify tau selection",
                fontsize=10, fontweight="bold")
ax_hz.legend(fontsize=9, frameon=False)
ax_hz.spines["top"].set_visible(False)
ax_hz.spines["right"].set_visible(False)

fig2.suptitle(
    f"Tau selection justification — MOSAIC GBM cohort (N={len(df)})\n"
    f"Median OS (events) = {median_os:.2f} years",
    fontsize=11, fontweight="bold")
fig2.tight_layout()
fig2.savefig(f"{SAVE_DIR}/tau_selection_justification.png",
             dpi=300, bbox_inches="tight")
fig2.savefig(f"{SAVE_DIR}/tau_selection_justification.pdf",
             bbox_inches="tight")
plt.show()
plt.close(fig2)
print("Saved tau_selection_justification.png/.pdf")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "="*60)
print("PROPORTIONAL HAZARDS ASSESSMENT SUMMARY")
print("="*60)
print(f"\nN={len(df)}, events={int(df['event'].sum())}, "
      f"median OS={median_os:.2f}y")
print(f"\nSchoenfeld residuals test:")
for var, p in [("MGMT",   mgmt_zph_p),
               ("Age",    age_zph_p),
               ("Sex",    sex_zph_p),
               ("Global", global_zph_p)]:
    flag = "*** PH VIOLATED" if p < 0.05 else "(not rejected)"
    print(f"  {var:<8}: p = {p:.4f}  {flag}")
print(f"\nConclusion:")
if age_zph_p < 0.05 or global_zph_p < 0.05:
    print(f"  PH assumption violated for at least one covariate.")
    print(f"  RMST is the more appropriate survival estimand for this cohort.")
else:
    print(f"  Formal Schoenfeld test does not reject PH, but visual")
    print(f"  diagnostics should be inspected for time-varying patterns.")
print(f"\nTau selection:")
print(f"  tau=3y  : inflection in population hazard; plateau in KM curves")
print(f"  tau=1.5y: approximate median OS ({median_os:.2f}y), "
      f"period of maximal event accrual")

### Load single cell data

In [ ]:
data_center = AllDataCenters # or "CHUV", "UKER", "CR"
adata_sc = AllDataCenters.load_singlecell(
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = False
)

### Create lognorm and UMAP layers for merged object

In [ ]:
# create a LogNormalize layer
adata_sc.layers["LogNormalize"] = adata_sc.layers["unnormalized_counts"].copy()
sc.pp.normalize_total(
    adata_sc, layer="LogNormalize", target_sum=1e4
)
sc.pp.log1p(adata_sc, layer="LogNormalize")

In [ ]:
USE_LAYER     = "LogNormalize"
USE_HVGS      = True
HVG_N_TOP     = 3000
N_PCS         = 50
N_NEIGHBORS   = 15
UMAP_MIN_DIST = 0.3
UMAP_SPREAD   = 1.0
RANDOM_STATE  = 13

adata_sc.obsm.pop("X_umap", None)
adata_sc.uns.pop("neighbors", None)
adata_sc.X = adata_sc.layers[USE_LAYER].copy()

if USE_HVGS:
    sc.pp.highly_variable_genes(adata_sc, n_top_genes=HVG_N_TOP, flavor="seurat_v3", subset=True)

sc.tl.pca(adata_sc, n_comps=N_PCS, svd_solver="arpack")
sc.pp.neighbors(adata_sc, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS)
sc.tl.umap(adata_sc, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD, random_state=RANDOM_STATE)

### Save lognorm and UMAP layer

In [ ]:
SAVE_DIR = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Final_Notebooks"
LAYER = "LogNormalize"
UMAP_KEY = "X_umap"

import os, numpy as np, pandas as pd, scipy.sparse as sp

os.makedirs(SAVE_DIR, exist_ok=True)

M = adata_sc.layers[LAYER]
M = M.tocsr() if sp.issparse(M) else sp.csr_matrix(M)
sp.save_npz(f"{SAVE_DIR}/{LAYER}.npz", M)

pd.Series(adata_sc.obs_names, name="obs").to_csv(f"{SAVE_DIR}/obs_names_v2.csv", index=False)
pd.Series(adata_sc.var_names, name="var").to_csv(f"{SAVE_DIR}/var_names_v2.csv", index=False)

U = adata_sc.obsm[UMAP_KEY]
pd.DataFrame(U, index=adata_sc.obs_names, columns=["UMAP1","UMAP2"]).to_csv(f"{SAVE_DIR}/umap_coords_v2.csv")

### Load lognorm and UMAP layers

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp

LOAD_DIR = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Final_Notebooks"
LAYER    = "LogNormalize"
UMAP_KEY = "X_umap"

# File checks
npz_path = f"{LOAD_DIR}/{LAYER}.npz"
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"Missing: {npz_path}")

for fn in ("obs_names_v2.csv", "var_names_v2.csv", "umap_coords_v2.csv"):
    p = f"{LOAD_DIR}/{fn}"
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}")

# Load saved artifacts
obs_saved = pd.read_csv(f"{LOAD_DIR}/obs_names_v2.csv")["obs"].astype(str).values
var_saved = pd.read_csv(f"{LOAD_DIR}/var_names_v2.csv")["var"].astype(str).values
M_saved = sp.load_npz(npz_path).tocsr()
U_saved = pd.read_csv(f"{LOAD_DIR}/umap_coords_v2.csv", index_col=0)

# Shape checks
if M_saved.shape != (len(obs_saved), len(var_saved)):
    raise ValueError(f"M_saved shape {M_saved.shape} != ({len(obs_saved)}, {len(var_saved)})")

# Build lookup dicts for fast indexing
obs_saved_idx = {name: i for i, name in enumerate(obs_saved)}
var_saved_idx = {name: i for i, name in enumerate(var_saved)}

# Current object
obs_cur = adata_sc.obs_names.astype(str).values
var_cur = adata_sc.var_names.astype(str).values

# Find overlapping indices
obs_cur_keep = []
obs_sav_keep = []
for i, name in enumerate(obs_cur):
    if name in obs_saved_idx:
        obs_cur_keep.append(i)
        obs_sav_keep.append(obs_saved_idx[name])

var_cur_keep = []
var_sav_keep = []
for i, name in enumerate(var_cur):
    if name in var_saved_idx:
        var_cur_keep.append(i)
        var_sav_keep.append(var_saved_idx[name])

obs_cur_keep = np.array(obs_cur_keep)
obs_sav_keep = np.array(obs_sav_keep)
var_cur_keep = np.array(var_cur_keep)
var_sav_keep = np.array(var_sav_keep)

if len(obs_cur_keep) == 0:
    raise ValueError("No overlapping obs_names")
if len(var_cur_keep) == 0:
    raise ValueError("No overlapping var_names")

# Extract submatrix and rebuild
sub = M_saved[obs_sav_keep, :][:, var_sav_keep].tocoo()
M_cur = sp.csr_matrix(
    (sub.data, (obs_cur_keep[sub.row], var_cur_keep[sub.col])),
    shape=(len(obs_cur), len(var_cur))
)
adata_sc.layers[LAYER] = M_cur

# Align UMAP
U_cur = np.full((len(obs_cur), 2), np.nan, dtype=float)
U_saved.index = U_saved.index.astype(str)
obs_in_umap = U_saved.index.isin(obs_cur)
U_sub = U_saved.loc[obs_in_umap].iloc[:, :2]
umap_idx = {name: i for i, name in enumerate(obs_cur)}
for name, row in U_sub.iterrows():
    if name in umap_idx:
        U_cur[umap_idx[name], :] = row.values[:2]
adata_sc.obsm[UMAP_KEY] = U_cur

print("Loaded into adata_sc:")
print(f"  layer: {LAYER} {adata_sc.layers[LAYER].shape} nnz: {adata_sc.layers[LAYER].nnz}")
print(f"  umap: {UMAP_KEY} {adata_sc.obsm[UMAP_KEY].shape} nan_rows: {np.isnan(adata_sc.obsm[UMAP_KEY]).any(axis=1).sum()}")
print(f"  obs_overlap: {len(obs_cur_keep)} / {len(obs_cur)}")
print(f"  var_overlap: {len(var_cur_keep)} / {len(var_cur)}")

### Load custom annotations for single cell

In [ ]:
import os
import pandas as pd

# user vars
ADATA_OBJ = adata_sc  # set to adata_sc (single cell) or adata (visium), explicitly
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"

LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

if not os.path.exists(KEY_PATH):
    raise FileNotFoundError(f"Missing KEY_PATH: {KEY_PATH}")

key = pd.read_csv(KEY_PATH)

# checks
if LEVEL4_KEY not in ADATA_OBJ.obs.columns:
    raise KeyError(f"Missing {LEVEL4_KEY} in ADATA_OBJ.obs")
missing_cols = [c for c in [LEVEL4_KEY] + NEW_COLS if c not in key.columns]
if missing_cols:
    raise KeyError(f"Missing column(s) in key file: {missing_cols}")

# normalize + deduplicate key
key = key.copy()
key[LEVEL4_KEY] = key[LEVEL4_KEY].astype(str).str.strip()
for c in NEW_COLS:
    key[c] = key[c].astype(str).where(key[c].notna(), None)

key = key.drop_duplicates(subset=[LEVEL4_KEY], keep="first").copy()

# mapping dicts from level4 -> curated levels
key_map = key.set_index(LEVEL4_KEY)[NEW_COLS].to_dict(orient="series")

# apply mappings to create new obs columns
src = ADATA_OBJ.obs[LEVEL4_KEY].astype(str).str.strip()

for col in NEW_COLS:
    mapped = src.map(key_map[col])
    out = mapped.fillna("Unmapped").astype(str)

    cats = pd.Index(key[col].dropna().astype(str).unique())
    if "Unmapped" in out.values and "Unmapped" not in cats:
        cats = cats.append(pd.Index(["Unmapped"]))

    ADATA_OBJ.obs[col] = pd.Categorical(out, categories=cats)

# quick report
lvl4_all = pd.Index(src.unique())
lvl4_in_key = pd.Index(key[LEVEL4_KEY].astype(str).unique())
missing = lvl4_all.difference(lvl4_in_key)

print(f"Target object: {'adata_sc' if ADATA_OBJ is adata_sc else 'adata'}")
print(f"Added columns: {NEW_COLS}")
print(f"Level4 labels in target: {len(lvl4_all)}")
print(f"Level4 labels covered by key: {len(lvl4_all.intersection(lvl4_in_key))}")
print(f"Level4 labels missing from key: {len(missing)}")
if len(missing):
    print("Unmapped labels:", ", ".join(list(missing)[:20]) + (" ..." if len(missing) > 20 else ""))

### Load Visium data (no relapse samples)

# Load raw_count for DEG analysis, log_norm_count for ligand-receptor analysis

adata = AllDataCenters.load_spatial_adata(
    exclude_low_quality_visium=True,
    exclude_idh1_mut=True,
    exclude_patient_ids=True,
    exclude_relapse=False,
    list_sample_id=None,
    normalization="raw_count",
    radius=1,
    deconvolution_slot_name="deconv_cohort_level_4",
)

print(adata)

### Load Visium data (with relapse samples)

In [ ]:
# Load full Visium cohort including relapse samples, excluding 4 broken UKER samples
import gbmcollaboration.definitions as defs

# 4 UKER recurrents fail zarr initialization (list index out of range)
BROKEN_RELAPSE = ["ER_G_298b", "ER_G_301b", "ER_G_645b", "ER_G_665b"]

_orig_relapse = defs.RELAPSE_EXCLUSION
defs.RELAPSE_EXCLUSION = BROKEN_RELAPSE

try:
    adata = AllDataCenters.load_spatial_adata(
        exclude_low_quality_visium=True,
        exclude_idh1_mut=True,
        exclude_patient_ids=True,
        exclude_relapse=True,        # now excludes only the 4 broken samples
        list_sample_id=None,
        normalization="raw_count",
        radius=1,
        deconvolution_slot_name="deconv_cohort_level_4",
    )
finally:
    defs.RELAPSE_EXCLUSION = _orig_relapse

print(adata)
print(f"Samples: {adata.obs['sample'].nunique()}, Spots: {adata.n_obs}")

### Load C2L total abundance data

In [ ]:
# Load ALL C2L abundance data (CHUV, UKER, Charite) - COMBINED
import gc
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor

def load_sample(center_name, sample_id, h5ad_file):
    """Load and process a single sample using Cell2LocationLoader approach."""
    try:
        if isinstance(h5ad_file, S3Path) or str(h5ad_file).startswith('s3://'):
            s3 = s3fs.S3FileSystem(anon=False)
            h5ad_path_str = str(h5ad_file).replace('s3://', '')
            with s3.open(h5ad_path_str, 'rb') as f:
                adata = ad.read_h5ad(f)
        else:
            adata = ad.read_h5ad(h5ad_file)
        
        if 'means_cell_abundance_w_sf' not in adata.obsm:
            return None
        
        celltype_names = None
        if 'mod' in adata.uns and 'factor_names' in adata.uns['mod']:
            celltype_names = adata.uns['mod']['factor_names']
        elif 'factor_names' in adata.uns:
            celltype_names = adata.uns['factor_names']
        elif 'cell_type_names' in adata.uns:
            celltype_names = adata.uns['cell_type_names']
        
        if celltype_names is None:
            n_celltypes = adata.obsm['means_cell_abundance_w_sf'].shape[1]
            celltype_names = [f'celltype_{i}' for i in range(n_celltypes)]
        
        celltype_names = [f'meanscell_abundance_w_sf_{name}' for name in celltype_names]
        
        cell_abundance = adata.obsm['means_cell_abundance_w_sf']
        if issparse(cell_abundance):
            cell_abundance = cell_abundance.toarray()
        
        df = pd.DataFrame(cell_abundance, index=adata.obs_names, columns=celltype_names)
        df = df.fillna(0)
        df.index = pd.MultiIndex.from_tuples(
            [(sample_id, idx) for idx in df.index],
            names=['sample', 'barcode']
        )
        df['total_cells'] = df.sum(axis=1)
        
        return df
    except Exception as e:
        return None


def load_center(center_name, cell2loc_path):
    """Load one center with original fast parallel, then kill workers."""
    print(f"\nProcessing {center_name}...")
    base_path = S3Path(str(cell2loc_path))
    sample_dirs = [d.name for d in base_path.iterdir() if d.is_dir()]
    print(f"  Found {len(sample_dirs)} sample dirs for {center_name}")
    
    samples_to_process = [
        (center_name, sample_id, base_path / sample_id / "cohort_level_4.h5ad")
        for sample_id in sample_dirs
    ]
    
    # Original fast parallel call
    center_results = Parallel(n_jobs=-1)(
        delayed(load_sample)(center_name, sample_id, h5ad_file)
        for center_name, sample_id, h5ad_file in tqdm(samples_to_process, desc=f"  {center_name}")
    )
    
    center_results = [df for df in center_results if df is not None]
    
    for df in center_results:
        new_index = pd.MultiIndex.from_tuples(
            [(center_name, *idx) for idx in df.index],
            names=["center", "sample", "barcode"],
        )
        df.index = new_index
        df["center"] = center_name
    
    df_center = pd.concat(center_results, axis=0)
    
    print(f"  Loaded {df_center.index.get_level_values('sample').nunique()} samples from {center_name}")
    print(f"  Total spots: {df_center.shape[0]}")
    
    # FORCE KILL parallel workers to prevent hang on next call
    get_reusable_executor().shutdown(wait=True)
    gc.collect()
    
    return df_center


# Load each center (workers killed between each)
df_chuv = load_center("CHUV", DATA_PATH_CHUV_FILES["Cell2location"])
df_uker = load_center("UKER", DATA_PATH_UKER_FILES["Cell2location"])
df_cr = load_center("CR", DATA_PATH_CR_FILES["Cell2location"])


# Merge all centers
df_all = pd.concat([df_chuv, df_uker, df_cr], axis=0)

print("\n" + "=" * 60)
print("Combined df_all summary:")
print("  Total spots:", df_all.shape[0])
print("  Total samples:", df_all.index.get_level_values("sample").nunique())
print("  Index levels:", df_all.index.names)
print("  Example columns:", df_all.columns[:5].tolist())
print("=" * 60)

### Merge C2L abundance data with Visium data

In [ ]:
# Drop the 'center' level from df_all's index for alignment
if isinstance(df_all.index, pd.MultiIndex) and "center" in df_all.index.names:
    df_all_no_center = df_all.copy()
    df_all_no_center.index = df_all_no_center.index.droplevel("center")
else:
    df_all_no_center = df_all

print("df_all_no_center index names:", df_all_no_center.index.names)
print("df_all_no_center shape:", df_all_no_center.shape)

# Build (sample, barcode) keys for each obs row, in obs order
barcodes = pd.Index(adata.obs_names).str.split("_").str[-1]
keys = pd.MultiIndex.from_arrays(
    [adata.obs["sample"].values, barcodes.values],
    names=["sample", "barcode"],
)

print("First 3 keys:", list(keys[:3]))
print("Number of keys:", len(keys))

# Align df_all_no_center to these keys
df_all_aligned = df_all_no_center.reindex(keys)
print("df_all_aligned shape:", df_all_aligned.shape)

# Drop columns from df_all_aligned that already exist in adata.obs
cols_to_add = [c for c in df_all_aligned.columns if c not in adata.obs.columns]
df_to_add = df_all_aligned[cols_to_add].copy()
df_to_add.index = adata.obs.index

# Join instead of concat to avoid fragmentation
adata.obs = adata.obs.join(df_to_add)

print(f"\nNumber of added columns: {len(cols_to_add)}")
print(f"Skipped (already exist): {len(df_all_aligned.columns) - len(cols_to_add)}")
print(f"adata.obs shape: {adata.obs.shape}")

### Load custom cell annotations for Visium

In [ ]:
import pandas as pd

# user vars
KEY_PATH = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/Analysis_Repo/Branch_infercnv_visium_backup/analysis/SingleCell_Annotation_Key.csv"
LEVEL4_KEY = "celltype_level4_scanvi"
NEW_COLS = [
    "celltype_level4_curated",
    "celltype_level3_curated",
    "celltype_level3_figure",
    "celltype_level2_curated",
    "celltype_level2_figure",
    "celltype_level1_curated",
]

# 1) Load key
key_raw = pd.read_csv(KEY_PATH)

# 2) Basic column checks
required_cols = [LEVEL4_KEY] + NEW_COLS
missing_cols = [c for c in required_cols if c not in key_raw.columns]
if missing_cols:
    raise ValueError(f"Missing columns in key file: {missing_cols}")

# 3) Deduplicate on the level4 key and standardize to string
key_unique = key_raw.drop_duplicates(subset=[LEVEL4_KEY]).copy()
key_unique[LEVEL4_KEY] = key_unique[LEVEL4_KEY].astype(str)

# 4) Build a tidy DataFrame indexed by level4
celltype_key = key_unique.set_index(LEVEL4_KEY)[NEW_COLS]
celltype_key = celltype_key.apply(lambda s: s.astype(str))

# 5) Attach as the canonical celltype hierarchy in adata.uns
adata.uns["celltype_annotation_key"] = celltype_key

# 6) Quick coverage checks vs Visium abundance columns
lvl4_in_key = pd.Index(celltype_key.index)

obs_cols = pd.Index(adata.obs.columns)

# deconv_cohort_level_4 columns should use the raw level4 names
deconv_level4_cols = obs_cols.intersection(lvl4_in_key)

# C2L totals use "meanscell_abundance_w_sf_<level4>"
c2l_cols = [c for c in obs_cols if c.startswith("meanscell_abundance_w_sf_")]
c2l_lvl4 = pd.Index([c.replace("meanscell_abundance_w_sf_", "") for c in c2l_cols])
c2l_in_key = c2l_lvl4.intersection(lvl4_in_key)

missing_for_deconv = lvl4_in_key.difference(deconv_level4_cols)
missing_for_c2l = lvl4_in_key.difference(c2l_in_key)

print("Stored celltype_annotation_key in adata.uns['celltype_annotation_key']")
print(f"Total level4 types in key: {len(lvl4_in_key)}")
print(f"Matching deconv level4 columns in adata.obs: {len(deconv_level4_cols)}")
print(f"Matching C2L level4 types (from meanscell_abundance_w_sf_*): {len(c2l_in_key)}")

if len(missing_for_deconv):
    print(f"Level4 types in key not seen as deconv columns: {len(missing_for_deconv)}")
if len(missing_for_c2l):
    print(f"Level4 types in key not seen in C2L columns: {len(missing_for_c2l)}")

### Load TIL spot metadata from file

In [ ]:
import pandas as pd
from pathlib import Path

# paths
EFS_ROOT = Path("/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd")
IN_CSV   = EFS_ROOT / "results" / "visium" / "til_zones" / "til_zone_annotations.csv"

# options
overwrite = True

# load
usecols = ["spot_id", "sample", "TIL_positive", "TIL_zone"]
df = pd.read_csv(
    IN_CSV,
    usecols=usecols,
    dtype={"spot_id": "string", "sample": "string", "TIL_zone": "string"},
)
df["spot_id"] = df["spot_id"].astype(str)
df = df.set_index("spot_id", drop=True)

# align
obs_names = pd.Index(adata.obs_names.astype(str))
df_aligned = df.reindex(obs_names)

# validate
if df_aligned["TIL_zone"].isna().any():
    n_missing = int(df_aligned["TIL_zone"].isna().sum())
    raise ValueError(f"{n_missing} adata.obs_names not found in {IN_CSV.name}")

if "sample" in adata.obs.columns:
    adata_sample = adata.obs["sample"].astype(str)
    csv_sample = df_aligned["sample"].astype(str)
    mism = (adata_sample != csv_sample) & csv_sample.notna()
    print("sample_mismatch_n:", int(mism.sum()))

# add
new_cols = ["TIL_positive", "TIL_zone"]
for c in new_cols:
    if (c in adata.obs.columns) and (not overwrite):
        raise KeyError(f"adata.obs['{c}'] already exists (set overwrite=True to replace)")

adata.obs["TIL_positive"] = df_aligned["TIL_positive"].astype("boolean")
til_order = ["TIL-", "BZ2", "BZ1", "TIL+"]
til_levels = [x for x in til_order if x in set(df_aligned["TIL_zone"].dropna().unique())]
if len(til_levels) >= 2:
    adata.obs["TIL_zone"] = pd.Categorical(df_aligned["TIL_zone"], categories=til_levels, ordered=True)
else:
    adata.obs["TIL_zone"] = df_aligned["TIL_zone"].astype("string")

# report
print("added_cols:", new_cols)
print("TIL_positive_counts:\n", adata.obs["TIL_positive"].value_counts(dropna=False))
print("TIL_zone_counts:\n", adata.obs["TIL_zone"].value_counts(dropna=False))

### Define TIL-low and TIL-high patients

In [ ]:
import pandas as pd
import numpy as np

VISIUM_SAMPLE_COL = "sample"
SC_SAMPLE_COL     = "orig.ident"
TIL_FLAG_COL      = "TIL_positive"

QUANTILE_CUTOFF = 0.75
MIN_TIL_SPOTS_FOR_HIGH = 0
MIN_TOTAL_SPOTS = 0
MIN_TIL_FRACTION_FOR_HIGH = None

KEY_LEN = 9

if "adata" not in globals():
    raise ValueError("adata (Visium) not found in the session.")
if "adata_sc" not in globals():
    raise ValueError("adata_sc (single-cell) not found in the session.")

for col in [VISIUM_SAMPLE_COL, TIL_FLAG_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing '{col}' in adata.obs")

if SC_SAMPLE_COL not in adata_sc.obs.columns:
    raise KeyError(f"Missing '{SC_SAMPLE_COL}' in adata_sc.obs")

visium_id_full = adata.obs[VISIUM_SAMPLE_COL].astype(str).str.replace(r"_vis$", "", regex=True)
visium_key = visium_id_full.str[:KEY_LEN]

df = (
    pd.DataFrame(
        {
            "join_key": visium_key.values,
            "visium_id_example": visium_id_full.values,
            "til_flag": adata.obs[TIL_FLAG_COL].astype(int).values,
        },
        index=adata.obs_names,
    )
    .groupby("join_key", as_index=False)
    .agg(
        TIL_spots=("til_flag", "sum"),
        Total_Spots=("til_flag", "size"),
        visium_id_example=("visium_id_example", lambda s: s.iloc[0]),
    )
)

df["TIL_fraction"] = df["TIL_spots"] / df["Total_Spots"]
df["TIL_percent"] = 100.0 * df["TIL_fraction"]

if MIN_TOTAL_SPOTS > 0:
    df = df[df["Total_Spots"] >= int(MIN_TOTAL_SPOTS)].copy()

if df.empty:
    raise ValueError("No groups remain after applying MIN_TOTAL_SPOTS filter.")

cutoff = float(df["TIL_fraction"].quantile(float(QUANTILE_CUTOFF)))
if MIN_TIL_FRACTION_FOR_HIGH is not None:
    cutoff = max(cutoff, float(MIN_TIL_FRACTION_FOR_HIGH))

df["TIL_group"] = "TIL_low"
high_mask = df["TIL_fraction"] >= cutoff
if MIN_TIL_SPOTS_FOR_HIGH > 0:
    high_mask &= df["TIL_spots"] >= int(MIN_TIL_SPOTS_FOR_HIGH)
df.loc[high_mask, "TIL_group"] = "TIL_high"

key_to_label = df.set_index("join_key")["TIL_group"].to_dict()

adata.obs["TIL_group"] = visium_key.map(key_to_label)
adata.obs["TIL_group"] = pd.Categorical(
    adata.obs["TIL_group"],
    categories=["TIL_high", "TIL_low"],
    ordered=True,
)

sc_id_full = adata_sc.obs[SC_SAMPLE_COL].astype(str)
sc_key = sc_id_full.str[:KEY_LEN]

adata_sc.obs["sample_id"] = sc_key.astype(str)

sc_labels = sc_key.map(key_to_label).fillna("Unmapped")
adata_sc.obs["TIL_group"] = pd.Categorical(
    sc_labels,
    categories=["TIL_high", "TIL_low", "Unmapped"],
    ordered=False,
)

adata.uns["TIL_group_stats"] = df.set_index("join_key")

print(f"Join key: first {KEY_LEN} chars of Visium sample (after stripping '_vis') and of single-cell orig.ident")
print(f"TIL_fraction cutoff for TIL_high (Q={QUANTILE_CUTOFF:.2f}): {cutoff:.6f}")

print("\nVisium group counts (unique join keys):")
print(df["TIL_group"].value_counts().to_string())

print("\nSingle-cell TIL_group value counts:")
print(adata_sc.obs["TIL_group"].value_counts(dropna=False).to_string())

n_sc_mapped = int((adata_sc.obs["TIL_group"].astype(str) != "Unmapped").sum())
print(f"\nSingle-cell mapped (not Unmapped): {n_sc_mapped} / {adata_sc.n_obs}")

vis_keys = set(pd.unique(visium_key))
sc_keys = set(pd.unique(sc_key))
only_in_sc = sorted(sc_keys - vis_keys)
only_in_vis = sorted(vis_keys - sc_keys)

print(f"\nKey coverage (unique join keys):")
print(f"  keys in Visium: {len(vis_keys)}")
print(f"  keys in scRNA : {len(sc_keys)}")
print(f"  keys only in scRNA (unmappable from Visium): {len(only_in_sc)}")
print(f"  keys only in Visium (no scRNA): {len(only_in_vis)}")
print(f"  examples only_in_sc (up to 20): {only_in_sc[:20]}")
print(f"  examples only_in_vis (up to 20): {only_in_vis[:20]}")

### Save data objects to local EFS

In [ ]:
import os

EXPORT_DIR = "/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/export_download"
os.makedirs(EXPORT_DIR, exist_ok=True)

# --- Clinical (tiny) ---
clin_all.to_parquet(f"{EXPORT_DIR}/clinical_all.parquet", index=False)
clin_all.to_csv(f"{EXPORT_DIR}/clinical_all.csv", index=False)
print(f"Clinical: {os.path.getsize(f'{EXPORT_DIR}/clinical_all.parquet') / 1e6:.1f} MB")

# --- Visium (medium-large) ---
print("Writing Visium adata...")
adata.write_h5ad(f"{EXPORT_DIR}/adata_visium_full.h5ad")
print(f"Visium: {os.path.getsize(f'{EXPORT_DIR}/adata_visium_full.h5ad') / 1e9:.2f} GB")

# --- Single Cell (largest) ---
print("Writing single-cell adata...")
adata_sc.write_h5ad(f"{EXPORT_DIR}/adata_sc_full.h5ad")
print(f"Single cell: {os.path.getsize(f'{EXPORT_DIR}/adata_sc_full.h5ad') / 1e9:.2f} GB")

# --- Summary ---
print("\n" + "=" * 60)
print("ALL FILES SAVED TO EFS:")
for f in sorted(os.listdir(EXPORT_DIR)):
    size = os.path.getsize(f"{EXPORT_DIR}/{f}")
    unit = "GB" if size > 1e9 else "MB"
    val = size / 1e9 if size > 1e9 else size / 1e6
    print(f"  {f}: {val:.2f} {unit}")
print(f"\nPath: {EXPORT_DIR}")
print("=" * 60)

# Questions from Manuscript

### Check ECM gene expression

In [ ]:
ecm_genes_brain_updated = [
    # Collagens
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1",  # restored per request
    "COL8A1", "COL10A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL16A1", "COL18A1",

    # ECM glycoproteins
    "FN1", "FBN1", "FBN2",
    "TNC", "TNR", "TNXB",
    "SPARC", "POSTN", "SPP1",
    "EFEMP1",  # added (GBM ECM literature)
    "LAMA1", "LAMA2", "LAMA3", "LAMA4", "LAMA5",  # LAMA3 restored
    "LAMB1", "LAMB2", "LAMB3",  # LAMB3 restored
    "LAMC1", "LAMC2", "LAMC3",  # LAMC2 restored
    "THBS1", "THBS2", "THBS4",
    "VTN",  # restored per request
    "NID1", "NID2",
    "FBLN1", "FBLN2", "FBLN5",
    "MMRN2",  # added (vascular ECM)
    "VWA1",   # added (ECM/secreted; glioma datasets)
    "TGFBI",  # added (GBM assoc with ECM organization)
    "LTBP1",  # added (malignant glioma ECM/TGFb activation)

    # Proteoglycans
    "VCAN", "ACAN", "BCAN", "NCAN",
    "DCN", "BGN", "LUM", "PRELP", "FMOD", "ASPN",
    "HSPG2", "AGRN",
    "GPC1", "GPC2", "GPC3",
    "SDC1", "SDC4",
    "CSPG4", "CSPG5", "PTPRZ1",
    "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",

    # Proteases
    "MMP1", "MMP2", "MMP3", "MMP7", "MMP9", "MMP11", "MMP12", "MMP13", "MMP14", "MMP16",
    "ADAMTS1", "ADAMTS4", "ADAMTS5", "ADAMTS8",
    "ADAM8",
    "HPSE",
    "PLAU", "PLAT",

    # Protease inhibitors
    "TIMP1", "TIMP2", "TIMP3",
    "SERPINE1",

    # Crosslinking / collagen maturation enzymes
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4",
    "TGM2",
    "P4HA1", "P4HA2", "P4HA3",
    "PLOD1", "PLOD2", "PLOD3",
    "BMP1",    # added (GBM-linked; collagen maturation biology)
    "PCOLCE",  # added (glioma-supported)

    # Hyaluronan metabolism
    "HAS1", "HAS2", "HAS3",
    "HYAL1", "HYAL2",
]

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import issparse

ecm_genes_brain_updated = [
    "COL1A1", "COL1A2", "COL3A1", "COL4A1", "COL4A2",
    "COL5A1", "COL5A2", "COL5A3", "COL6A1", "COL6A2", "COL6A3",
    "COL7A1", "COL8A1", "COL10A1", "COL11A1", "COL12A1",
    "COL14A1", "COL15A1", "COL16A1", "COL18A1",
    "FN1", "FBN1", "FBN2", "TNC", "TNR", "TNXB",
    "SPARC", "POSTN", "SPP1", "EFEMP1",
    "LAMA1", "LAMA2", "LAMA3", "LAMA4", "LAMA5",
    "LAMB1", "LAMB2", "LAMB3", "LAMC1", "LAMC2", "LAMC3",
    "THBS1", "THBS2", "THBS4", "VTN", "NID1", "NID2",
    "FBLN1", "FBLN2", "FBLN5", "MMRN2", "VWA1", "TGFBI", "LTBP1",
    "VCAN", "ACAN", "BCAN", "NCAN", "DCN", "BGN", "LUM", "PRELP",
    "FMOD", "ASPN", "HSPG2", "AGRN", "GPC1", "GPC2", "GPC3",
    "SDC1", "SDC4", "CSPG4", "CSPG5", "PTPRZ1",
    "HAPLN1", "HAPLN2", "HAPLN3", "HAPLN4",
    "MMP1", "MMP2", "MMP3", "MMP7", "MMP9", "MMP11", "MMP12",
    "MMP13", "MMP14", "MMP16", "ADAMTS1", "ADAMTS4", "ADAMTS5",
    "ADAMTS8", "ADAM8", "HPSE", "PLAU", "PLAT",
    "TIMP1", "TIMP2", "TIMP3", "SERPINE1",
    "LOX", "LOXL1", "LOXL2", "LOXL3", "LOXL4", "TGM2",
    "P4HA1", "P4HA2", "P4HA3", "PLOD1", "PLOD2", "PLOD3",
    "BMP1", "PCOLCE",
    "HAS1", "HAS2", "HAS3", "HYAL1", "HYAL2",
]

MIN_PATIENTS = 5

# --- Check gene presence in both datasets ---
vis_genes = set(adata.var_names)
sc_genes = set(adata_sc.var_names)

not_in_vis = [g for g in ecm_genes_brain_updated if g not in vis_genes]
not_in_sc = [g for g in ecm_genes_brain_updated if g not in sc_genes]
print(f"Genes not in Visium var_names: {not_in_vis if not_in_vis else 'None'}")
print(f"Genes not in SC var_names:     {not_in_sc if not_in_sc else 'None'}")

# --- Visium: mean expression per patient, then count patients with meaningful expression ---
# Use raw_count layer; threshold = mean raw count > 0.5 per spot in a patient
vis_ecm = [g for g in ecm_genes_brain_updated if g in vis_genes]
gene_idx_vis = [list(adata.var_names).index(g) for g in vis_ecm]

sample_ids = adata.obs['sample'].values
unique_samples = np.unique(sample_ids)

# Build patient-level mean expression
X_vis = adata.layers['raw_count']
results_vis = []

for g, gi in zip(vis_ecm, gene_idx_vis):
    col = X_vis[:, gi]
    if issparse(col):
        col = col.toarray().ravel()
    else:
        col = np.asarray(col).ravel()
    
    # Mean per sample
    patient_means = pd.Series(col, index=adata.obs_names).groupby(sample_ids).mean()
    n_expressing = (patient_means > 0.5).sum()
    overall_mean = col.mean()
    pct_nonzero = (col > 0).mean() * 100
    
    results_vis.append({
        'gene': g, 'overall_mean': overall_mean, 'pct_nonzero_spots': pct_nonzero,
        'n_patients_expressing': n_expressing
    })

df_vis = pd.DataFrame(results_vis).sort_values('n_patients_expressing')

# --- Single cell: same logic, use LogNormalize layer ---
sc_ecm = [g for g in ecm_genes_brain_updated if g in sc_genes]
gene_idx_sc = [list(adata_sc.var_names).index(g) for g in sc_ecm]

sc_sample_ids = adata_sc.obs['sample_id'].values

X_sc = adata_sc.layers['LogNormalize']
results_sc = []

for g, gi in zip(sc_ecm, gene_idx_sc):
    col = X_sc[:, gi]
    if issparse(col):
        col = col.toarray().ravel()
    else:
        col = np.asarray(col).ravel()
    
    patient_means = pd.Series(col, index=adata_sc.obs_names).groupby(sc_sample_ids).mean()
    n_expressing = (patient_means > 0.1).sum()  # LogNorm threshold
    overall_mean = col.mean()
    pct_nonzero = (col > 0).mean() * 100
    
    results_sc.append({
        'gene': g, 'overall_mean': overall_mean, 'pct_nonzero_cells': pct_nonzero,
        'n_patients_expressing': n_expressing
    })

df_sc = pd.DataFrame(results_sc).sort_values('n_patients_expressing')

# --- Merge and flag ---
df_merged = df_vis.rename(columns=lambda c: f'vis_{c}' if c != 'gene' else c).merge(
    df_sc.rename(columns=lambda c: f'sc_{c}' if c != 'gene' else c),
    on='gene', how='outer'
)

df_merged['PASS_vis'] = df_merged['vis_n_patients_expressing'] >= MIN_PATIENTS
df_merged['PASS_sc'] = df_merged['sc_n_patients_expressing'] >= MIN_PATIENTS
df_merged['PASS_both'] = df_merged['PASS_vis'] & df_merged['PASS_sc']

# --- Report ---
fail = df_merged[~df_merged['PASS_both']].sort_values('vis_n_patients_expressing')
passing = df_merged[df_merged['PASS_both']]

print(f"\n{'='*90}")
print(f"RESULTS: {len(passing)} genes PASS, {len(fail)} genes FAIL (< {MIN_PATIENTS} patients in either modality)")
print(f"{'='*90}")

if len(fail):
    print(f"\nFAILING GENES:")
    print(fail[['gene', 'vis_n_patients_expressing', 'vis_pct_nonzero_spots', 'vis_overall_mean',
                'sc_n_patients_expressing', 'sc_pct_nonzero_cells', 'sc_overall_mean',
                'PASS_vis', 'PASS_sc']].to_string(index=False))

print(f"\nLOWEST-EXPRESSING PASSING GENES (bottom 15):")
bottom = passing.sort_values('vis_n_patients_expressing').head(15)
print(bottom[['gene', 'vis_n_patients_expressing', 'vis_pct_nonzero_spots',
              'sc_n_patients_expressing', 'sc_pct_nonzero_cells']].to_string(index=False))

In [ ]:
# Check candidate additions
candidates = [
    'LGALS1',   # Galectin-1, heavily expressed in GBM ECM, promotes invasion/immunosuppression
    'LGALS3',   # Galectin-3, ECM-binding lectin, GBM progression marker
    'CCN2',     # CTGF, secreted ECM organizer, fibrosis/angiogenesis in GBM
    'CCN1',     # CYR61, matricellular, integrin signaling
    'SEMA3C',   # Secreted semaphorin, ECM-organizing role in GBM invasion
    'CTSD',     # Cathepsin D, secreted protease, GBM ECM remodeling
    'CTSB',     # Cathepsin B, secreted protease, GBM invasion
    'IGFBP7',   # Secreted, ECM-associated, in your MES-hypoxia signature
    'CHI3L1',   # YKL-40, secreted glycoprotein, ECM/inflammation, GBM hallmark
    'FLRT2',    # Fibronectin leucine-rich, brain ECM
    'SLIT2',    # Secreted, axon guidance/ECM patterning in brain
    'EMILIN1',  # Elastic fiber-associated, vascular ECM
    'MFGE8',    # Secreted, integrin ligand, brain ECM
    'ITIH5',    # Inter-alpha-trypsin inhibitor, brain ECM matrisome
]

# Quick expression check
for dset, obj, layer, thresh, sample_col in [
    ('Visium', adata, 'raw_count', 0.5, 'sample'),
    ('SC', adata_sc, 'LogNormalize', 0.1, 'sample_id'),
]:
    present = [g for g in candidates if g in obj.var_names]
    missing = [g for g in candidates if g not in obj.var_names]
    if missing:
        print(f"\n{dset} - not in var_names: {missing}")
    
    X = obj.layers[layer]
    samples = obj.obs[sample_col].values
    
    for g in present:
        gi = list(obj.var_names).index(g)
        col = X[:, gi]
        if issparse(col):
            col = col.toarray().ravel()
        else:
            col = np.asarray(col).ravel()
        patient_means = pd.Series(col).groupby(samples).mean()
        n_expr = (patient_means > thresh).sum()
        pct_nz = (col > 0).mean() * 100
        print(f"  {dset:6s}  {g:10s}  n_patients={n_expr:4d}  pct_nonzero={pct_nz:.1f}%")

### Total and relative differential TIL abundance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from scipy import sparse
import gc

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
OUT_DIR = '.'

T_NK_CELLTYPES = ['NK_KLRC1', 'NK_nos', 'T_CD4_nos', 'T_CD8_nos', 
                  'T_IFN_activated', 'T_NK_cycling', 'T_nos', 'T_reg']

print("=" * 80)
print("T/NK CELL ANALYSIS: TIL_high vs TIL_low")
print("=" * 80)
print(f"\nAnalyzing {len(T_NK_CELLTYPES)} T/NK cell types")

# PART 1: VISIUM CELL ABUNDANCE
print("\n" + "=" * 80)
print("VISIUM - T/NK CELL ABUNDANCE")
print("=" * 80)

t_nk_c2l_cols = [f'meanscell_abundance_w_sf_{ct}' for ct in T_NK_CELLTYPES]
available_cols = [col for col in t_nk_c2l_cols if col in adata.obs.columns]
available_types = [col.replace('meanscell_abundance_w_sf_', '') for col in available_cols]

print(f"\nFound {len(available_cols)}/{len(T_NK_CELLTYPES)} cell types in Visium data")

til_mask = adata.obs['TIL_group'].notna()
high_mask = til_mask & (adata.obs['TIL_group'] == 'TIL_high')
low_mask = til_mask & (adata.obs['TIL_group'] == 'TIL_low')

print(f"Spots: TIL_high={high_mask.sum():,}, TIL_low={low_mask.sum():,}")

visium_results = []
for celltype, col in zip(available_types, available_cols):
    high = adata.obs.loc[high_mask, col].values
    low = adata.obs.loc[low_mask, col].values
    high_total = adata.obs.loc[high_mask, 'total_cells'].values
    low_total = adata.obs.loc[low_mask, 'total_cells'].values
    
    stat, pval = mannwhitneyu(high, low, alternative='two-sided')
    
    visium_results.append({
        'celltype': celltype,
        'TIL_high_mean': high.mean(),
        'TIL_low_mean': low.mean(),
        'fold_change': high.mean() / low.mean() if low.mean() > 0 else np.inf,
        'TIL_high_rel': (high / high_total).mean(),
        'TIL_low_rel': (low / low_total).mean(),
        'p_value': pval,
        'significant': pval < 0.05
    })

df_visium = pd.DataFrame(visium_results)
df_visium['log2_fc'] = np.log2(df_visium['fold_change'])
print("\n", df_visium[['celltype', 'TIL_high_mean', 'TIL_low_mean', 'fold_change', 'p_value', 'significant']])
df_visium.to_csv(f'{OUT_DIR}/visium_tcell_nk_abundance.csv', index=False)

if 'T_IFN_activated' in df_visium['celltype'].values:
    ifn = df_visium[df_visium['celltype'] == 'T_IFN_activated'].iloc[0]
    print(f"\nT_IFN_activated: {ifn['fold_change']:.2f}x higher in TIL_high, p={ifn['p_value']:.2e}")

del high, low, high_total, low_total, stat, pval
gc.collect()

# PART 2: SINGLE-CELL CELL ABUNDANCE
print("\n" + "=" * 80)
print("SINGLE-CELL - T/NK CELL ABUNDANCE")
print("=" * 80)

sc_til_mask = adata_sc.obs['TIL_group'].isin(['TIL_high', 'TIL_low'])
print(f"\nCells: Total={sc_til_mask.sum():,}, TIL_high={(adata_sc.obs['TIL_group']=='TIL_high').sum():,}, TIL_low={(adata_sc.obs['TIL_group']=='TIL_low').sum():,}")

key = adata.uns['celltype_annotation_key']
scanvi_to_curated = key['celltype_level4_curated'].to_dict()
curated_to_scanvi = {v: k for k, v in scanvi_to_curated.items()}

t_nk_mask = sc_til_mask & (adata_sc.obs['celltype_level2_curated'] == 'T_NK')
print(f"T/NK cells: {t_nk_mask.sum():,}")

sc_obs_subset = adata_sc.obs.loc[t_nk_mask, ['sample_id', 'celltype_level4_curated', 'TIL_group']].copy()
sc_obs_subset['scanvi_name'] = sc_obs_subset['celltype_level4_curated'].map(curated_to_scanvi)
sc_obs_subset = sc_obs_subset[sc_obs_subset['scanvi_name'].isin(T_NK_CELLTYPES)]

print(f"T/NK cells matching our {len(T_NK_CELLTYPES)} types: {len(sc_obs_subset):,}")

patient_summary = adata_sc.obs.loc[sc_til_mask, ['sample_id', 'TIL_group']].groupby(['sample_id', 'TIL_group'], observed=True).size().reset_index(name='total_cells')

t_nk_counts = sc_obs_subset.groupby(['sample_id', 'scanvi_name'], observed=True).size().reset_index(name='count')
t_nk_pivot = t_nk_counts.pivot(index='sample_id', columns='scanvi_name', values='count').fillna(0)
t_nk_pivot = t_nk_pivot.merge(patient_summary, left_index=True, right_on='sample_id', how='left')

count_cols = [col for col in t_nk_pivot.columns if col in T_NK_CELLTYPES]
for col in count_cols:
    t_nk_pivot[f'{col}_per10k'] = (t_nk_pivot[col] / t_nk_pivot['total_cells']) * 10000

sc_results = []
for celltype in T_NK_CELLTYPES:
    col = f'{celltype}_per10k'
    if col not in t_nk_pivot.columns:
        continue
    
    high_vals = t_nk_pivot.loc[t_nk_pivot['TIL_group'] == 'TIL_high', col].values
    low_vals = t_nk_pivot.loc[t_nk_pivot['TIL_group'] == 'TIL_low', col].values
    
    if len(high_vals) > 0 and len(low_vals) > 0:
        stat, pval = mannwhitneyu(high_vals, low_vals, alternative='two-sided')
        sc_results.append({
            'celltype': celltype,
            'TIL_high_mean': high_vals.mean(),
            'TIL_low_mean': low_vals.mean(),
            'fold_change': high_vals.mean() / low_vals.mean() if low_vals.mean() > 0 else np.inf,
            'p_value': pval,
            'significant': pval < 0.05
        })

df_sc = pd.DataFrame(sc_results)
df_sc['log2_fc'] = np.log2(df_sc['fold_change'])
df_sc = df_sc.sort_values('p_value')
print("\n", df_sc[['celltype', 'TIL_high_mean', 'TIL_low_mean', 'fold_change', 'p_value', 'significant']])
df_sc.to_csv(f'{OUT_DIR}/singlecell_tcell_nk_abundance.csv', index=False)

if 'T_IFN_activated' in df_sc['celltype'].values:
    ifn = df_sc[df_sc['celltype'] == 'T_IFN_activated'].iloc[0]
    print(f"\nT_IFN_activated: {ifn['fold_change']:.2f}x higher in TIL_high, p={ifn['p_value']:.2e}")

del sc_obs_subset, patient_summary, t_nk_counts, t_nk_pivot, high_vals, low_vals, stat, pval
gc.collect()

# PART 3: GENE EXPRESSION
print("\n" + "=" * 80)
print("GENE EXPRESSION - CD3E, CD4, CD8A")
print("=" * 80)

GENES_OF_INTEREST = ['CD3E', 'CD4', 'CD8A']

print("\nVISIUM:")
visium_gene_results = []
for gene in GENES_OF_INTEREST:
    if gene not in adata.var_names:
        print(f"  {gene}: NOT FOUND")
        continue
    
    gene_idx = adata.var_names.get_loc(gene)
    high_expr = adata.layers['raw_count'][high_mask, gene_idx]
    low_expr = adata.layers['raw_count'][low_mask, gene_idx]
    
    if sparse.issparse(high_expr):
        high_expr = high_expr.toarray().flatten()
    if sparse.issparse(low_expr):
        low_expr = low_expr.toarray().flatten()
    
    stat, pval = mannwhitneyu(high_expr, low_expr, alternative='two-sided')
    
    visium_gene_results.append({
        'gene': gene,
        'TIL_high_mean': high_expr.mean(),
        'TIL_low_mean': low_expr.mean(),
        'fold_change': high_expr.mean() / low_expr.mean() if low_expr.mean() > 0 else np.inf,
        'log2_fc': np.log2(high_expr.mean() / low_expr.mean()) if low_expr.mean() > 0 else np.nan,
        'p_value': pval,
        'significant': pval < 0.05
    })
    
    del high_expr, low_expr

df_vis_genes = pd.DataFrame(visium_gene_results)
print(df_vis_genes)
df_vis_genes.to_csv(f'{OUT_DIR}/visium_gene_expression.csv', index=False)

print("\nSINGLE-CELL:")
sc_gene_results = []
for gene in GENES_OF_INTEREST:
    if gene not in adata_sc.var_names:
        print(f"  {gene}: NOT FOUND")
        continue
    
    gene_idx = adata_sc.var_names.get_loc(gene)
    high_idx = (sc_til_mask & (adata_sc.obs['TIL_group'] == 'TIL_high')).values
    low_idx = (sc_til_mask & (adata_sc.obs['TIL_group'] == 'TIL_low')).values
    
    high_expr = adata_sc.layers['LogNormalize'][high_idx, gene_idx]
    low_expr = adata_sc.layers['LogNormalize'][low_idx, gene_idx]
    
    if sparse.issparse(high_expr):
        high_expr = high_expr.toarray().flatten()
    if sparse.issparse(low_expr):
        low_expr = low_expr.toarray().flatten()
    
    stat, pval = mannwhitneyu(high_expr, low_expr, alternative='two-sided')
    
    sc_gene_results.append({
        'gene': gene,
        'TIL_high_mean': high_expr.mean(),
        'TIL_low_mean': low_expr.mean(),
        'fold_change': high_expr.mean() / low_expr.mean() if low_expr.mean() > 0 else np.inf,
        'log2_fc': np.log2(high_expr.mean() / low_expr.mean()) if low_expr.mean() > 0 else np.nan,
        'p_value': pval,
        'significant': pval < 0.05
    })
    
    del high_expr, low_expr

df_sc_genes = pd.DataFrame(sc_gene_results)
print(df_sc_genes)
df_sc_genes.to_csv(f'{OUT_DIR}/singlecell_gene_expression.csv', index=False)

del stat, pval, high_idx, low_idx
gc.collect()

# VISUALIZATION
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
df_plot = df_visium.sort_values('log2_fc', ascending=True)
colors = ['#d62728' if p < 0.05 else '#cccccc' for p in df_plot['p_value']]
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['log2_fc'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['celltype'], fontsize=9)
ax.set_xlabel('Log2 FC (TIL_high / TIL_low)', fontsize=10)
ax.set_title('VISIUM: T/NK Cell Abundance', fontsize=11, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)

ax = axes[0, 1]
df_plot = df_sc.sort_values('log2_fc', ascending=True)
colors = ['#d62728' if p < 0.05 else '#cccccc' for p in df_plot['p_value']]
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['log2_fc'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['celltype'], fontsize=9)
ax.set_xlabel('Log2 FC (TIL_high / TIL_low)', fontsize=10)
ax.set_title('SINGLE-CELL: T/NK Cell Abundance', fontsize=11, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)

ax = axes[1, 0]
df_plot = df_vis_genes.sort_values('log2_fc', ascending=True)
colors = ['#d62728' if p < 0.05 else '#cccccc' for p in df_plot['p_value']]
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['log2_fc'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['gene'], fontsize=10)
ax.set_xlabel('Log2 FC (TIL_high / TIL_low)', fontsize=10)
ax.set_title('VISIUM: Gene Expression', fontsize=11, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)

ax = axes[1, 1]
df_plot = df_sc_genes.sort_values('log2_fc', ascending=True)
colors = ['#d62728' if p < 0.05 else '#cccccc' for p in df_plot['p_value']]
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['log2_fc'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['gene'], fontsize=10)
ax.set_xlabel('Log2 FC (TIL_high / TIL_low)', fontsize=10)
ax.set_title('SINGLE-CELL: Gene Expression', fontsize=11, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/tcell_nk_analysis_combined.png', dpi=300, bbox_inches='tight')
print(f"\nSaved: {OUT_DIR}/tcell_nk_analysis_combined.png")
plt.close()

del df_plot, colors, y_pos, ax, axes, fig
gc.collect()

print("\nDONE")

### Samples per modality

In [ ]:
import pandas as pd
import numpy as np

print("=" * 100)
print("PATIENT COUNTS BY CENTER AND MODALITY")
print("=" * 100)

# Map center names between modalities
CENTER_MAP = {
    'CR': 'Charite',
    'CHUV': 'CHUV',
    'UKER': 'UKER'
}

# Standardize center names
def standardize_center(center):
    if center == 'CR':
        return 'Charite'
    return center

# CLINICAL DATA
print("\n--- CLINICAL DATA ---")
print(f"Total rows: {len(clin_all)}")
print(f"Unique patients (Subject Id): {clin_all['Subject Id'].nunique()}")

clinical_by_center = clin_all.groupby('center')['Subject Id'].nunique()
print("\nBy center:")
for center, count in clinical_by_center.items():
    print(f"  {center:10s}: {count:3d} patients")

# VISIUM DATA  
print("\n--- VISIUM DATA ---")

# Get unique sample names (not spots!)
visium_samples = adata.obs['sample'].str.replace('_vis$', '', regex=True).unique()
print(f"Unique Visium samples: {len(visium_samples)}")

# Extract patient ID (first 8 chars) and center from each sample
visium_data = []
for sample in visium_samples:
    patient_id = sample[:8]
    # Infer center from ID prefix
    if patient_id.startswith('CH_'):
        center = 'CHUV'
    elif patient_id.startswith('ER_'):
        center = 'UKER'
    elif patient_id.startswith('CR_'):
        center = 'Charite'
    else:
        center = 'Unknown'
    visium_data.append({'sample': sample, 'patient_id': patient_id, 'center': center})

df_visium = pd.DataFrame(visium_data)

# Count unique patients per center
visium_by_center = df_visium.groupby('center')['patient_id'].nunique()
print("\nBy center:")
for center, count in visium_by_center.items():
    print(f"  {center:10s}: {count:3d} patients")

print(f"\nSamples per patient (should see 1-2 typically):")
samples_per_patient = df_visium.groupby('patient_id').size().value_counts().sort_index()
for n_samples, n_patients in samples_per_patient.head(10).items():
    print(f"  {n_samples} sample(s): {n_patients} patients")

# SINGLE-CELL DATA
print("\n--- SINGLE-CELL DATA ---")

# Extract patient ID (first 8 chars) from sample_id
sc_patient_ids = adata_sc.obs['sample_id'].str[:8].unique()
print(f"Unique single-cell patients: {len(sc_patient_ids)}")

# Infer center from ID prefix
sc_data = []
for patient_id in sc_patient_ids:
    if patient_id.startswith('CH_'):
        center = 'CHUV'
    elif patient_id.startswith('ER_'):
        center = 'UKER'
    elif patient_id.startswith('CR_'):
        center = 'Charite'
    else:
        center = 'Unknown'
    sc_data.append({'patient_id': patient_id, 'center': center})

df_sc = pd.DataFrame(sc_data)

sc_by_center = df_sc.groupby('center')['patient_id'].nunique()
print("\nBy center:")
for center, count in sc_by_center.items():
    print(f"  {center:10s}: {count:3d} patients")

# CROSS-MODALITY OVERLAP
print("\n" + "=" * 100)
print("CROSS-MODALITY PATIENT OVERLAP BY CENTER")
print("=" * 100)

centers = ['CHUV', 'UKER', 'Charite']

for center in centers:
    print(f"\n{center}:")
    
    # Get patient sets for this center
    clin_pts = set(clin_all[clin_all['center'] == center]['Subject Id'].unique())
    vis_pts = set(df_visium[df_visium['center'] == center]['patient_id'].unique())
    sc_pts = set(df_sc[df_sc['center'] == center]['patient_id'].unique())
    
    print(f"  Clinical:     {len(clin_pts):3d} patients")
    print(f"  Visium:       {len(vis_pts):3d} patients")
    print(f"  Single-cell:  {len(sc_pts):3d} patients")
    
    # Overlaps
    clin_vis = len(clin_pts & vis_pts)
    clin_sc = len(clin_pts & sc_pts)
    vis_sc = len(vis_pts & sc_pts)
    all_three = len(clin_pts & vis_pts & sc_pts)
    
    # Only in one modality
    clin_only = len(clin_pts - vis_pts - sc_pts)
    vis_only = len(vis_pts - clin_pts - sc_pts)
    sc_only = len(sc_pts - clin_pts - vis_pts)
    
    print(f"\n  Overlaps:")
    print(f"    Clinical + Visium:      {clin_vis:3d}")
    print(f"    Clinical + SC:          {clin_sc:3d}")
    print(f"    Visium + SC:            {vis_sc:3d}")
    print(f"    All three modalities:   {all_three:3d}")
    
    print(f"\n  Unique to modality:")
    print(f"    Clinical only:          {clin_only:3d}")
    print(f"    Visium only:            {vis_only:3d}")
    print(f"    SC only:                {sc_only:3d}")

# SUMMARY TABLE
print("\n" + "=" * 100)
print("SUMMARY TABLE")
print("=" * 100)

summary_data = []
for center in centers:
    clin_count = clinical_by_center.get(center, 0)
    vis_count = visium_by_center.get(center, 0)
    sc_count = sc_by_center.get(center, 0)
    
    summary_data.append({
        'Center': center,
        'Clinical': clin_count,
        'Visium': vis_count,
        'Single-cell': sc_count
    })

summary_data.append({
    'Center': 'TOTAL',
    'Clinical': clin_all['Subject Id'].nunique(),
    'Visium': df_visium['patient_id'].nunique(),
    'Single-cell': df_sc['patient_id'].nunique()
})

df_summary = pd.DataFrame(summary_data)
print("\n")
print(df_summary.to_string(index=False))

df_summary.to_csv('./patient_counts_corrected.csv', index=False)
print(f"\nSummary saved to: ./patient_counts_corrected.csv")

# OVERALL MODALITY COMBINATIONS
print("\n" + "=" * 100)
print("OVERALL MODALITY COMBINATIONS (across all centers)")
print("=" * 100)

all_clin = set(clin_all['Subject Id'].unique())
all_vis = set(df_visium['patient_id'].unique())
all_sc = set(df_sc['patient_id'].unique())
all_patients = all_clin | all_vis | all_sc

modality_counts = {
    'Clinical only': len(all_clin - all_vis - all_sc),
    'Visium only': len(all_vis - all_clin - all_sc),
    'SC only': len(all_sc - all_clin - all_vis),
    'Clinical + Visium (no SC)': len((all_clin & all_vis) - all_sc),
    'Clinical + SC (no Visium)': len((all_clin & all_sc) - all_vis),
    'Visium + SC (no Clinical)': len((all_vis & all_sc) - all_clin),
    'All three': len(all_clin & all_vis & all_sc)
}

print(f"\nTotal unique patients: {len(all_patients)}")
print("\nBy modality combination:")
for combo, count in modality_counts.items():
    pct = 100 * count / len(all_patients) if len(all_patients) > 0 else 0
    print(f"  {combo:30s}: {count:3d} ({pct:5.1f}%)")

print("\n" + "=" * 100)

### Range in overall survival

In [ ]:
from lifelines import KaplanMeierFitter

# Use deduplicated clinical data (one row per patient, 'a' block preferred)
clin_os = clin_dedup[clin_dedup['os_years'].notna() & clin_dedup['os_event'].notna()].copy()

print(f"N patients: {len(clin_os)}, Deaths: {int(clin_os['os_event'].sum())}, "
      f"Censored: {int((clin_os['os_event'] == 0).sum())}")
print(f"OS range: {clin_os['os_years'].min():.3f} - {clin_os['os_years'].max():.3f} years")

# Overall KM
kmf = KaplanMeierFitter()
kmf.fit(clin_os['os_years'], clin_os['os_event'], label='All')
print(f"\nMedian OS (KM): {kmf.median_survival_time_:.3f} years")

# By center
print("\nBy center:")
for center in sorted(clin_os['center'].unique()):
    sub = clin_os[clin_os['center'] == center]
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(sub['os_years'], sub['os_event'])
    print(f"  {center:8s}  n={len(sub):3d}  events={int(sub['os_event'].sum()):3d}  median_OS={kmf_c.median_survival_time_:.3f} yr")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

kmf.plot_survival_function(ax=ax1, ci_show=True, linewidth=2)
ax1.axhline(0.5, color='red', ls='--', lw=1, alpha=0.5)
ax1.set(xlabel='Years', ylabel='Survival probability', title=f'Overall Survival (n={len(clin_os)}, median={kmf.median_survival_time_:.2f} yr)')

for center in sorted(clin_os['center'].unique()):
    sub = clin_os[clin_os['center'] == center]
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(sub['os_years'], sub['os_event'], label=f'{center} (n={len(sub)})')
    kmf_c.plot_survival_function(ax=ax2, ci_show=False, linewidth=2)

ax2.set(xlabel='Years', ylabel='Survival probability', title='OS by Center')
plt.tight_layout()
plt.show()

### Marker genes for malignant cell types

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.sparse import issparse

# --- Config ---
MAX_CELLS_PER_TYPE = 5000
SEED = 42
SUBTYPES = [
    'Malignant_AC1', 'Malignant_AC2', 'Malignant_AC-like_IFN_response',
    'Malignant_OPC1', 'Malignant_OPC2', 'Malignant_OPC-like_invasive',
    'Malignant_NPC2',
    'Malignant_MES1', 'Malignant_MES-hypoxia',
    'Malignant_cycling',
]

# Parent Neftel state mapping
NEFTEL_PARENT = {
    'Malignant_AC1': 'AC', 'Malignant_AC2': 'AC', 'Malignant_AC-like_IFN_response': 'AC',
    'Malignant_OPC1': 'OPC', 'Malignant_OPC2': 'OPC', 'Malignant_OPC-like_invasive': 'OPC',
    'Malignant_NPC2': 'NPC',
    'Malignant_MES1': 'MES', 'Malignant_MES-hypoxia': 'MES',
    'Malignant_cycling': None,
}

# Neftel 2019 signatures
NEFTEL_SIGS = {
    'AC': {'GFAP','SLC1A3','SLC1A2','GLUL','AGT','AQP4','GJA1','CLU','HOPX','CD44',
           'VIM','S100B','ALDH1L1','ID3','HEPN1','SLC7A11','BCAN','NDRG2','EDNRB',
           'ATP1A2','NFIA','SOX9'},
    'OPC': {'OLIG1','OLIG2','SOX10','PDGFRA','CSPG4','GPR17','TNR','PLLP','APOD',
            'VCAN','BCAN','PTPRZ1','COL9A1','LHFPL3','MEGF11','SEMA5A','NEU4',
            'PMP2','CDH20','CNTN1'},
    'NPC': {'DCX','SOX4','SOX11','CD24','STMN1','STMN2','STMN4','DLX5','DLX6-AS1',
            'ASCL1','DLL3','NEUROD1','HES6','INSM1','NCAM1','TUBB3','MAPT','MAP2','ELAVL4'},
    'MES': {'CHI3L1','ANXA1','ANXA2','S100A4','S100A6','S100A10','S100A11','VIM','CD44',
            'TGFBI','SERPINE1','TIMP1','NNMT','INHBA','SFRP1','SFRP2','THBS1','COL1A1',
            'COL1A2','COL3A1','FN1','VEGFA','IL6','FOSL2','RELB','NFKB1','NFKB2'},
}

# --- Subset and subsample ---
mask = adata_sc.obs['celltype_level4_curated'].isin(SUBTYPES)
adata_mal = adata_sc[mask].copy()
adata_mal.X = adata_mal.layers['LogNormalize'].copy()

# Subsample per type
np.random.seed(SEED)
keep_idx = []
for st in SUBTYPES:
    idx = np.where(adata_mal.obs['celltype_level4_curated'] == st)[0]
    if len(idx) > MAX_CELLS_PER_TYPE:
        idx = np.random.choice(idx, MAX_CELLS_PER_TYPE, replace=False)
    keep_idx.append(idx)
keep_idx = np.concatenate(keep_idx)
adata_sub = adata_mal[keep_idx].copy()

print(f"Subsampled: {adata_sub.n_obs:,} cells from {len(SUBTYPES)} subtypes")
print(adata_sub.obs['celltype_level4_curated'].value_counts())

# --- DE (fast on subsampled data) ---
sc.tl.rank_genes_groups(
    adata_sub,
    groupby='celltype_level4_curated',
    method='wilcoxon',
    use_raw=False,
    n_genes=200,
)

# --- Extract results ---
print("\n" + "=" * 90)
print(f"{'Subtype':<38s} | {'Top 5 DE markers':<45s} | {'Top 5 in Neftel sig'}")
print("=" * 90)

all_results = []

for st in SUBTYPES:
    df_de = sc.get.rank_genes_groups_df(adata_sub, group=st)
    df_de = df_de[(df_de['pvals_adj'] < 0.01) & (df_de['logfoldchanges'] > 0.5)]
    
    # Top 5 data-driven
    top5_de = df_de.head(5)['names'].tolist()
    
    # Top 5 cross-referenced with parent Neftel signature
    parent = NEFTEL_PARENT[st]
    if parent is not None:
        sig_genes = NEFTEL_SIGS[parent]
        neftel_hits = df_de[df_de['names'].isin(sig_genes)].head(5)['names'].tolist()
    else:
        # For cycling: check all signatures
        all_sig = set().union(*NEFTEL_SIGS.values())
        neftel_hits = df_de[df_de['names'].isin(all_sig)].head(5)['names'].tolist()
    
    print(f"{st:<38s} | {', '.join(top5_de):<45s} | {', '.join(neftel_hits) if neftel_hits else 'None'}")
    
    all_results.append({
        'subtype': st,
        'neftel_parent': parent or 'Mixed',
        'top5_de': top5_de,
        'top5_neftel_xref': neftel_hits,
        'n_sig_genes': len(df_de),
    })

# --- Save ---
df_out = pd.DataFrame(all_results)
df_out['top5_de'] = df_out['top5_de'].apply(', '.join)
df_out['top5_neftel_xref'] = df_out['top5_neftel_xref'].apply(', '.join)
df_out.to_csv('./malignant_subtype_signatures.csv', index=False)
print(f"\nSaved: malignant_subtype_signatures.csv")

# Cleanup
del adata_mal, adata_sub

In [ ]:
from collections import Counter
all_de = [g for r in all_results for g in r['top5_de']]
dupes = {g: c for g, c in Counter(all_de).items() if c > 1}
print(f"Shared genes across data-driven top 5: {dupes if dupes else 'None — all unique'}")

### Clinical covariate survival analysis

In [ ]:
# Check lifelines
import subprocess
try:
    import lifelines
    print(f"lifelines version: {lifelines.__version__}")
except ImportError:
    subprocess.check_call(['pip', 'install', 'lifelines', '--break-system-packages'])
    import lifelines
    print(f"lifelines installed: {lifelines.__version__}")

In [ ]:
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

# --- Prepare data ---
df = clin_dedup[['os_years', 'os_event', 'age_at_diagnosis', 
                  'MGMT promoter methylation', 'p53', 'Administrative gender']].copy()
df = df[df['os_years'].notna() & df['os_event'].notna()].copy()

# Recode
df['MGMT'] = df['MGMT promoter methylation'].map({'Methylated': 'Methylated', 'Unmethylated': 'Unmethylated'})
df['p53'] = df['p53'].map({'Positive': 'Positive', 'Negative': 'Negative'})
df['Sex'] = df['Administrative gender']
df['Age_group'] = pd.cut(df['age_at_diagnosis'], bins=[0, 65, 200], labels=['≤65', '>65'])

# --- Multivariate Cox ---
cox_df = df[['os_years', 'os_event', 'age_at_diagnosis', 'MGMT', 'p53', 'Sex']].dropna().copy()
cox_df['MGMT_meth'] = (cox_df['MGMT'] == 'Methylated').astype(int)
cox_df['p53_pos'] = (cox_df['p53'] == 'Positive').astype(int)
cox_df['female'] = (cox_df['Sex'] == 'Female').astype(int)

cph = CoxPHFitter()
cph.fit(cox_df[['os_years', 'os_event', 'age_at_diagnosis', 'MGMT_meth', 'p53_pos', 'female']],
        duration_col='os_years', event_col='os_event')

print("=" * 80)
print("MULTIVARIATE COX PROPORTIONAL HAZARDS")
print("=" * 80)
cph.print_summary()

# --- KM + univariate Cox for each variable ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

def km_plot_binary(ax, data, group_col, time_col='os_years', event_col='os_event', title=''):
    """KM curves for a binary variable with logrank + univariate Cox stats."""
    clean = data[[time_col, event_col, group_col]].dropna()
    groups = sorted(clean[group_col].unique())
    
    colors = ['#FFA500', '#6495ED'] if len(groups) == 2 else plt.cm.tab10.colors
    
    for i, g in enumerate(groups):
        sub = clean[clean[group_col] == g]
        kmf = KaplanMeierFitter()
        kmf.fit(sub[time_col], sub[event_col], label=f'{g} (n={len(sub)})')
        kmf.plot_survival_function(ax=ax, ci_show=False, linewidth=2, color=colors[i])
    
    # Logrank
    g0 = clean[clean[group_col] == groups[0]]
    g1 = clean[clean[group_col] == groups[1]]
    lr = logrank_test(g0[time_col], g1[time_col], g0[event_col], g1[event_col])
    
    # Univariate Cox
    cox_tmp = clean.copy()
    cox_tmp['_x'] = (cox_tmp[group_col] == groups[1]).astype(int)
    cph_uni = CoxPHFitter()
    cph_uni.fit(cox_tmp[[time_col, event_col, '_x']], duration_col=time_col, event_col=event_col)
    hr = cph_uni.hazard_ratios_['_x']
    ci_lo = np.exp(cph_uni.confidence_intervals_.iloc[0, 0])
    ci_hi = np.exp(cph_uni.confidence_intervals_.iloc[0, 1])
    cox_p = cph_uni.summary['p']['_x']
    
    ax.set_title(f'{title}\nHR={hr:.2f} ({ci_lo:.2f}-{ci_hi:.2f}), Cox p={cox_p:.1e}, logrank p={lr.p_value:.1e}',
                 fontsize=11, fontweight='bold')
    ax.set(xlabel='Years', ylabel='Survival probability')
    ax.legend(fontsize=9, loc='lower left')
    ax.set_ylim([0, 1.05])
    ax.grid(False)

# 1) Age
km_plot_binary(axes[0, 0], df, 'Age_group', title='Age at Diagnosis (≤65 vs >65)')

# 2) MGMT
km_plot_binary(axes[0, 1], df, 'MGMT', title='MGMT Promoter Methylation')

# 3) p53
km_plot_binary(axes[1, 0], df, 'p53', title='p53 Status')

# 4) Sex
km_plot_binary(axes[1, 1], df, 'Sex', title='Sex')

plt.tight_layout()
plt.savefig('./cox_km_clinical_covariates.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nDone. Figure saved: ./cox_km_clinical_covariates.png")

In [ ]:
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

# --- Prepare data ---
df = clin_dedup[['os_years', 'os_event', 'age_at_diagnosis', 
                  'MGMT promoter methylation', 'p53', 'Administrative gender']].copy()
df = df[df['os_years'].notna() & df['os_event'].notna()].copy()

# Recode
df['MGMT'] = df['MGMT promoter methylation'].map({'Methylated': 'Methylated', 'Unmethylated': 'Unmethylated'})
df['p53'] = df['p53'].map({'Positive': 'Positive', 'Negative': 'Negative'})
df['Sex'] = df['Administrative gender']
df['Age_group'] = pd.cut(df['age_at_diagnosis'], bins=[0, 65, 200], labels=['≤65', '>65'])

# --- Multivariate Cox ---
cox_df = df[['os_years', 'os_event', 'age_at_diagnosis', 'MGMT', 'p53', 'Sex']].dropna().copy()
cox_df['MGMT_meth'] = (cox_df['MGMT'] == 'Methylated').astype(int)
cox_df['p53_pos'] = (cox_df['p53'] == 'Positive').astype(int)
cox_df['female'] = (cox_df['Sex'] == 'Female').astype(int)

cph = CoxPHFitter()
cph.fit(cox_df[['os_years', 'os_event', 'age_at_diagnosis', 'MGMT_meth', 'p53_pos', 'female']],
        duration_col='os_years', event_col='os_event')

print("=" * 80)
print("MULTIVARIATE COX PROPORTIONAL HAZARDS")
print("=" * 80)
cph.print_summary()

# =====================================================================
# CONFIG
# =====================================================================
CI_ALPHA        = 0.10          # 1 - confidence level (0.10 = 90% CI)
FIG_WIDTH       = 4.5           # per-panel width
FIG_HEIGHT      = 3           # per-panel height
GROUP_COLORS    = ['#ff7b02', '#007ebc']   # orange / blue

# ── Aesthetics ───────────────────────────────────────────────────────
TITLE_SIZE      = 10
AXIS_LABEL_SIZE = 12
TICK_SIZE        = 8
LEGEND_SIZE     = 12
STATS_SIZE      = 8
KM_LINEWIDTH    = 1
CI_BAND_ALPHA   = 0.15
SPINE_WIDTH     = 0.4
AXIS_LINEWIDTH  = 0.4
TICK_WIDTH       = 0.4
TICK_LENGTH      = 1.0
CENSOR_SIZE     = 4             # 0 to hide censor ticks

# --- KM plotting function ---
def km_plot_binary(data, group_col, time_col='os_years', event_col='os_event',
                   title='', save_stem=None, ci_alpha=CI_ALPHA):
    """KM curves for a binary variable — separate figure, CI bands, stats as legend."""
    clean = data[[time_col, event_col, group_col]].dropna()
    groups = sorted(clean[group_col].unique())

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    for i, g in enumerate(groups):
        sub = clean[clean[group_col] == g]
        kmf = KaplanMeierFitter()
        kmf.fit(sub[time_col], sub[event_col], label=f'{g} (n={len(sub)})',
                alpha=ci_alpha)
        kmf.plot_survival_function(ax=ax, ci_show=True,
                                   linewidth=KM_LINEWIDTH,
                                   color=GROUP_COLORS[i],
                                   ci_alpha=CI_BAND_ALPHA)
        # Censor marks
        if CENSOR_SIZE > 0:
            censored = sub[sub[event_col] == 0]
            surv_fn = kmf.survival_function_at_times(censored[time_col]).values
            ax.plot(censored[time_col], surv_fn, '|',
                    color=GROUP_COLORS[i], markersize=CENSOR_SIZE,
                    markeredgewidth=TICK_WIDTH)

    # Logrank
    g0 = clean[clean[group_col] == groups[0]]
    g1 = clean[clean[group_col] == groups[1]]
    lr = logrank_test(g0[time_col], g1[time_col], g0[event_col], g1[event_col])

    # Univariate Cox
    cox_tmp = clean.copy()
    cox_tmp['_x'] = (cox_tmp[group_col] == groups[1]).astype(int)
    cph_uni = CoxPHFitter()
    cph_uni.fit(cox_tmp[[time_col, event_col, '_x']],
                duration_col=time_col, event_col=event_col)
    hr = cph_uni.hazard_ratios_['_x']
    ci_lo = np.exp(cph_uni.confidence_intervals_.iloc[0, 0])
    ci_hi = np.exp(cph_uni.confidence_intervals_.iloc[0, 1])
    cox_p = cph_uni.summary['p']['_x']

    # Stats as legend entry
    stats_label = (f'HR={hr:.2f} ({ci_lo:.2f}–{ci_hi:.2f})\n'
                   f'Cox p={cox_p:.1e}\nLog-rank p={lr.p_value:.1e}')
    ax.plot([], [], ' ', label=stats_label)

    ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold')
    ax.set_xlabel('Years', fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel('Survival probability', fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_ylim([0, 1.05])
    ax.grid(False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.legend(fontsize=LEGEND_SIZE, loc='upper right', framealpha=0.8,
              handlelength=1.2, borderpad=0.4, labelspacing=0.3)

    fig.tight_layout()
    if save_stem:
        fig.savefig(f'{save_stem}.png', dpi=300, bbox_inches='tight')
        fig.savefig(f'{save_stem}.pdf', bbox_inches='tight')
        print(f'  Saved {save_stem}.png/pdf')
    plt.show()
    plt.close(fig)

# --- Generate individual plots ---
km_plot_binary(df, 'Age_group', title='Age at Diagnosis (≤65 vs >65)',
               save_stem='km_age')
km_plot_binary(df, 'MGMT', title='MGMT Promoter Methylation',
               save_stem='km_mgmt')
km_plot_binary(df, 'p53', title='p53 Status',
               save_stem='km_p53')
km_plot_binary(df, 'Sex', title='Sex',
               save_stem='km_sex')

print("\nDone. Individual KM figures saved.")

## RMST Survival Modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from lifelines import KaplanMeierFitter
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

# =============================================================================
# CONFIG
# =============================================================================

TIME_COL  = "os_years"
EVENT_COL = "os_event"
TAU       = 3.0

FIG_WIDTH  = 4.5
FIG_HEIGHT = 3.0
GROUP_COLORS_2  = ["#ff7b02", "#007ebc"]           # orange / blue (binary plots)
GROUP_COLORS_3  = ["#ff7b02", "#007ebc", "#2ca02c"] # orange / blue / green (center)
TITLE_SIZE      = 10
AXIS_LABEL_SIZE = 12
TICK_SIZE       = 8
LEGEND_SIZE     = 9
KM_LINEWIDTH    = 1
CI_BAND_ALPHA   = 0.15
SPINE_WIDTH     = 0.4
TICK_WIDTH      = 0.4
TICK_LENGTH     = 1.0
CENSOR_SIZE     = 4

SAVE_DIR = "."

# Site code mapping
SITE_LABELS = {"CH": "CHUV", "ER": "UKER", "CR": "Charité"}

# =============================================================================
# PREPARE DATA
# =============================================================================

clin_dedup = clin_out.copy()
clin_dedup["_priority"] = clin_dedup["Sample collection chronology"].map(
    {"Baseline": 0, "At recurrence": 1}
).fillna(2)
clin_dedup = clin_dedup.sort_values(["Subject Id", "_priority"])
clin_dedup = clin_dedup.drop_duplicates(subset=["Subject Id"], keep="first")
clin_dedup = clin_dedup.drop(columns=["_priority"])

df = clin_dedup[[
    TIME_COL, EVENT_COL,
    "age_at_diagnosis",
    "MGMT promoter methylation",
    "p53",
    "Administrative gender",
    "Site code",
]].copy()

df = df[df[TIME_COL].notna() & df[EVENT_COL].notna()].copy()
df["time"]  = pd.to_numeric(df[TIME_COL],  errors="coerce")
df["event"] = pd.to_numeric(df[EVENT_COL], errors="coerce")
df = df[df["time"] > 0].copy()

# Categorical groupings
df["MGMT"]      = df["MGMT promoter methylation"].map(
                      {"Methylated": "Methylated", "Unmethylated": "Unmethylated"})
df["p53"]       = df["p53"].map({"Positive": "Positive", "Negative": "Negative"})
df["Sex"]       = df["Administrative gender"]
df["Age_group"] = pd.cut(df["age_at_diagnosis"],
                          bins=[0, 65, 200], labels=["≤65", ">65"])
df["Center"]    = df["Site code"].astype(str)

# Numeric encodings for covariate adjustment
df["mgmt_num"] = (df["MGMT"] == "Methylated").astype(float)
df["p53_num"]  = (df["p53"] == "Positive").astype(float)
df["sex_num"]  = (df["Sex"] == "Female").astype(float)
df["age_z"]    = ((df["age_at_diagnosis"] - df["age_at_diagnosis"].mean()) /
                   df["age_at_diagnosis"].std())

# Center dummies (ER as reference, drop_first drops ER)
center_dummies = pd.get_dummies(df["Center"], prefix="center", drop_first=True).astype(float)
df = pd.concat([df, center_dummies], axis=1)
center_dummy_cols = center_dummies.columns.tolist()

print(f"Patients with complete survival data: {len(df)}")
print(f"Center dummy columns (ER is reference): {center_dummy_cols}")
print(df[["MGMT", "p53", "Sex", "Age_group", "Center"]].apply(
    lambda c: c.value_counts(dropna=False)).T)

# =============================================================================
# LOAD R PACKAGES
# =============================================================================

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")

# =============================================================================
# RMST HELPER
# =============================================================================

def run_rmst(df_arm1, df_arm0, tau, cov_cols=None):
    """
    arm1 = group of interest, arm0 = reference.
    Returns dict with unadjusted and (if cov_cols) adjusted results.
    """
    df_comb = pd.concat([
        df_arm0.assign(arm=0),
        df_arm1.assign(arm=1),
    ], ignore_index=True)
    df_comb = df_comb.dropna(subset=["time", "event"])
    df_comb = df_comb[df_comb["time"] > 0]

    if tau > df_comb["time"].max():
        print(f"  Warning: tau={tau} exceeds max follow-up {df_comb['time'].max():.2f}")
        return None
    if len(df_arm1) < 5 or len(df_arm0) < 5:
        print(f"  Warning: insufficient n (arm1={len(df_arm1)}, arm0={len(df_arm0)})")
        return None

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(df_comb)
    ro.globalenv["surv_data"] = r_df
    ro.globalenv["tau_val"]   = float(tau)

    try:
        ro.r("""
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result[1, "Est."]
        u_lo   <- rmst_u$unadjusted.result[1, "lower .95"]
        u_hi   <- rmst_u$unadjusted.result[1, "upper .95"]
        u_p    <- rmst_u$unadjusted.result[1, "p"]
        rmst_h <- rmst_u$RMST.arm1$rmst["Est."]
        rmst_l <- rmst_u$RMST.arm0$rmst["Est."]
        """)

        out = {
            "n_arm1":     len(df_arm1),
            "n_arm0":     len(df_arm0),
            "tau":        tau,
            "rmst_arm1":  float(ro.r("rmst_h")[0]),
            "rmst_arm0":  float(ro.r("rmst_l")[0]),
            "unadj_diff": float(ro.r("u_diff")[0]),
            "unadj_lo":   float(ro.r("u_lo")[0]),
            "unadj_hi":   float(ro.r("u_hi")[0]),
            "unadj_p":    float(ro.r("u_p")[0]),
        }

        avail_covs = [c for c in (cov_cols or [])
                      if c in df_comb.columns and df_comb[c].notna().all()]
        if avail_covs:
            cov_r = df_comb[avail_covs].values.astype(float)
            ro.globalenv["cov_mat"] = ro.r.matrix(
                ro.FloatVector(cov_r.flatten(order="C")),
                nrow=int(cov_r.shape[0]),
                ncol=int(cov_r.shape[1]),
                byrow=True,
            )
            ro.r("""
            rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                            tau=tau_val, covariates=cov_mat)
            a_diff <- rmst_a$adjusted.result[1, "Est."]
            a_lo   <- rmst_a$adjusted.result[1, "lower .95"]
            a_hi   <- rmst_a$adjusted.result[1, "upper .95"]
            a_p    <- rmst_a$adjusted.result[1, "p"]
            """)
            out.update({
                "adj_diff": float(ro.r("a_diff")[0]),
                "adj_lo":   float(ro.r("a_lo")[0]),
                "adj_hi":   float(ro.r("a_hi")[0]),
                "adj_p":    float(ro.r("a_p")[0]),
            })

        return out

    except Exception as e:
        print(f"  survRM2 error: {e}")
        return None

# =============================================================================
# BINARY KM + RMST PLOT
# =============================================================================

def km_rmst_plot(data, group_col, arm1_label, arm0_label,
                 title="", save_stem=None, tau=TAU, cov_cols=None):
    cols_needed = list({TIME_COL, EVENT_COL, group_col} | set(cov_cols or []))
    clean = data[cols_needed].dropna(subset=[TIME_COL, EVENT_COL, group_col]).copy()
    clean["time"]  = pd.to_numeric(clean[TIME_COL],  errors="coerce")
    clean["event"] = pd.to_numeric(clean[EVENT_COL], errors="coerce")
    clean = clean[clean["time"] > 0]

    df_arm1 = clean[clean[group_col] == arm1_label].copy()
    df_arm0 = clean[clean[group_col] == arm0_label].copy()

    res = run_rmst(df_arm1, df_arm0, tau, cov_cols=cov_cols)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    for i, (sub, label) in enumerate([
        (df_arm0, f"{arm0_label} (n={len(df_arm0)})"),
        (df_arm1, f"{arm1_label} (n={len(df_arm1)})"),
    ]):
        kmf = KaplanMeierFitter()
        kmf.fit(sub["time"], sub["event"], label=label)
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            linewidth=KM_LINEWIDTH,
            color=GROUP_COLORS_2[i],
            ci_alpha=CI_BAND_ALPHA,
        )
        if CENSOR_SIZE > 0:
            censored = sub[sub["event"] == 0]
            surv_at_censor = kmf.survival_function_at_times(
                censored["time"]).values
            ax.plot(censored["time"], surv_at_censor, "|",
                    color=GROUP_COLORS_2[i],
                    markersize=CENSOR_SIZE,
                    markeredgewidth=TICK_WIDTH)

    ax.axvline(x=tau, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(tau + 0.05, 0.97, f"τ={int(tau)}y",
            fontsize=7, color="gray", va="top")

    if res:
        p_key   = "adj_p" if "adj_p" in res else "unadj_p"
        p_label = "adj p" if "adj_p" in res else "p"
        diff_mo = res["adj_diff"] * 12 if "adj_diff" in res else res["unadj_diff"] * 12
        stats_txt = (
            f"RMST diff: {diff_mo:+.1f} mo (τ={int(tau)}y)\n"
            f"{p_label} = {res[p_key]:.3f}"
        )
        ax.plot([], [], " ", label=stats_txt)

    ax.set_title(title, fontsize=TITLE_SIZE, fontweight="bold")
    ax.set_xlabel("Years", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Survival probability", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0)
    ax.grid(False)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["bottom", "left"]:
        ax.spines[sp].set_linewidth(SPINE_WIDTH)
    ax.legend(fontsize=LEGEND_SIZE, loc="upper right",
              framealpha=0.8, handlelength=1.2,
              borderpad=0.4, labelspacing=0.3)

    fig.tight_layout()
    if save_stem:
        fig.savefig(f"{save_stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(f"{save_stem}.pdf", bbox_inches="tight")
        print(f"  Saved {save_stem}.png/.pdf")
    plt.show()
    plt.close(fig)
    return res

# =============================================================================
# THREE-WAY CENTER KM PLOT
# =============================================================================

def km_center_plot(data, tau=TAU, cov_cols=None, save_stem=None):
    """
    KM plot with all three centers. Three pairwise RMST comparisons annotated.
    Centers: CH (orange), ER (blue), CR (green).
    """
    centers    = ["CH", "ER", "CR"]
    colors     = GROUP_COLORS_3
    cov_cols   = cov_cols or []

    cols_needed = list({TIME_COL, EVENT_COL, "Center"} | set(cov_cols))
    clean = data[cols_needed].dropna(subset=[TIME_COL, EVENT_COL, "Center"]).copy()
    clean["time"]  = pd.to_numeric(clean[TIME_COL],  errors="coerce")
    clean["event"] = pd.to_numeric(clean[EVENT_COL], errors="coerce")
    clean = clean[clean["time"] > 0]

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    subsets = {}
    for code, color in zip(centers, colors):
        sub = clean[clean["Center"] == code].copy()
        subsets[code] = sub
        label = f"{SITE_LABELS[code]} (n={len(sub)})"
        kmf = KaplanMeierFitter()
        kmf.fit(sub["time"], sub["event"], label=label)
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            linewidth=KM_LINEWIDTH,
            color=color,
            ci_alpha=CI_BAND_ALPHA,
        )
        if CENSOR_SIZE > 0:
            censored = sub[sub["event"] == 0]
            surv_at_censor = kmf.survival_function_at_times(
                censored["time"]).values
            ax.plot(censored["time"], surv_at_censor, "|",
                    color=color,
                    markersize=CENSOR_SIZE,
                    markeredgewidth=TICK_WIDTH)

    ax.axvline(x=tau, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(tau + 0.05, 0.97, f"τ={int(tau)}y",
            fontsize=7, color="gray", va="top")

    # Pairwise RMST: CH vs ER, CH vs CR, CR vs ER
    pairs = [("CH", "ER"), ("CH", "CR"), ("CR", "ER")]
    res_center = {}
    for arm1_code, arm0_code in pairs:
        key = f"{arm1_code}-{arm0_code}"
        res_center[key] = run_rmst(
            subsets[arm1_code], subsets[arm0_code],
            tau=tau, cov_cols=cov_cols
        )

    # Annotate pairwise p-values as legend entries
    for (arm1_code, arm0_code) in pairs:
        key = f"{arm1_code}-{arm0_code}"
        res = res_center[key]
        if res:
            p_key   = "adj_p" if "adj_p" in res else "unadj_p"
            p_label = "adj p" if "adj_p" in res else "p"
            label_str = (
                f"{SITE_LABELS[arm1_code]}–{SITE_LABELS[arm0_code]} "
                f"{p_label} = {res[p_key]:.3f}"
            )
        else:
            label_str = f"{SITE_LABELS[arm1_code]}–{SITE_LABELS[arm0_code]}: insufficient data"
        ax.plot([], [], " ", label=label_str)

    ax.set_title("Collection Center", fontsize=TITLE_SIZE, fontweight="bold")
    ax.set_xlabel("Years", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Survival probability", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0)
    ax.grid(False)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["bottom", "left"]:
        ax.spines[sp].set_linewidth(SPINE_WIDTH)
    ax.legend(fontsize=LEGEND_SIZE, loc="upper right",
              framealpha=0.8, handlelength=1.2,
              borderpad=0.4, labelspacing=0.3)

    fig.tight_layout()
    if save_stem:
        fig.savefig(f"{save_stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(f"{save_stem}.pdf", bbox_inches="tight")
        print(f"  Saved {save_stem}.png/.pdf")
    plt.show()
    plt.close(fig)
    return res_center

# =============================================================================
# GENERATE KM + RMST PLOTS
# =============================================================================

print("\n--- Age ---")
res_age = km_rmst_plot(
    df, "Age_group", arm1_label=">65", arm0_label="≤65",
    title="Age at Diagnosis (≤65 vs >65)",
    save_stem=f"{SAVE_DIR}/km_rmst_age",
    cov_cols=["mgmt_num", "p53_num", "sex_num"] + center_dummy_cols,
)

print("\n--- MGMT ---")
res_mgmt = km_rmst_plot(
    df, "MGMT", arm1_label="Methylated", arm0_label="Unmethylated",
    title="MGMT Promoter Methylation",
    save_stem=f"{SAVE_DIR}/km_rmst_mgmt",
    cov_cols=["p53_num", "sex_num", "age_z"] + center_dummy_cols,
)

print("\n--- p53 ---")
res_p53 = km_rmst_plot(
    df, "p53", arm1_label="Positive", arm0_label="Negative",
    title="p53 Status",
    save_stem=f"{SAVE_DIR}/km_rmst_p53",
    cov_cols=["mgmt_num", "sex_num", "age_z"] + center_dummy_cols,
)

print("\n--- Sex ---")
res_sex = km_rmst_plot(
    df, "Sex", arm1_label="Female", arm0_label="Male",
    title="Sex",
    save_stem=f"{SAVE_DIR}/km_rmst_sex",
    cov_cols=["mgmt_num", "p53_num", "age_z"] + center_dummy_cols,
)

print("\n--- Center ---")
res_center = km_center_plot(
    df, tau=TAU,
    cov_cols=["mgmt_num", "p53_num", "sex_num", "age_z"],
    save_stem=f"{SAVE_DIR}/km_rmst_center",
)

# =============================================================================
# FOREST PLOT (SUPPLEMENTARY)
# Uses matplotlib table for clean text layout — no text positioning arithmetic
# =============================================================================

def get_res(res, key="adj"):
    """Extract adjusted (or unadjusted fallback) diff/lo/hi/p from result dict."""
    if res is None:
        return np.nan, np.nan, np.nan, np.nan
    if f"{key}_diff" in res:
        return (res[f"{key}_diff"] * 12,
                res[f"{key}_lo"]   * 12,
                res[f"{key}_hi"]   * 12,
                res[f"{key}_p"])
    # fallback to unadjusted
    return (res["unadj_diff"] * 12,
            res["unadj_lo"]   * 12,
            res["unadj_hi"]   * 12,
            res["unadj_p"])

forest_rows = [
    ("MGMT methylated vs unmethylated", res_mgmt,
     res_mgmt["n_arm1"] if res_mgmt else np.nan,
     res_mgmt["n_arm0"] if res_mgmt else np.nan),
    ("Age >65 vs ≤65", res_age,
     res_age["n_arm1"]  if res_age  else np.nan,
     res_age["n_arm0"]  if res_age  else np.nan),
    ("Sex: Female vs Male", res_sex,
     res_sex["n_arm1"]  if res_sex  else np.nan,
     res_sex["n_arm0"]  if res_sex  else np.nan),
    ("p53 positive vs negative", res_p53,
     res_p53["n_arm1"]  if res_p53  else np.nan,
     res_p53["n_arm0"]  if res_p53  else np.nan),
]

# Unpack center pairwise results into separate rows
center_pairs = [
    ("Center: CHUV vs UKER",    res_center.get("CH-ER")),
    ("Center: CHUV vs Charité", res_center.get("CH-CR")),
    ("Center: Charité vs UKER", res_center.get("CR-ER")),
]
for label, res in center_pairs:
    n1 = res["n_arm1"] if res else np.nan
    n0 = res["n_arm0"] if res else np.nan
    forest_rows.append((label, res, n1, n0))

# Build records
records = []
for label, res, n1, n0 in forest_rows:
    diff, lo, hi, p = get_res(res)
    records.append({
        "label":   label,
        "n_comp":  int(n1) if not np.isnan(n1) else "—",
        "n_ref":   int(n0) if not np.isnan(n0) else "—",
        "diff_mo": diff,
        "lo_mo":   lo,
        "hi_mo":   hi,
        "p":       p,
    })

forest_df = pd.DataFrame(records)

# --- Plot ---
n_rows = len(forest_df)
fig_h  = 1.2 + n_rows * 0.55
fig, ax = plt.subplots(figsize=(9, fig_h))

y_positions = np.arange(n_rows)[::-1].astype(float)

for i, (_, row) in enumerate(forest_df.iterrows()):
    y   = y_positions[i]
    sig = (not np.isnan(row["p"])) and row["p"] < 0.05
    col = "#C0392B" if sig else "#2C3E50"

    if not np.isnan(row["diff_mo"]):
        ax.errorbar(
            x=row["diff_mo"], y=y,
            xerr=[[row["diff_mo"] - row["lo_mo"]],
                  [row["hi_mo"]   - row["diff_mo"]]],
            fmt="D", color=col,
            markersize=5, capsize=3,
            linewidth=1.3, elinewidth=1.1,
            zorder=3,
        )

# Axis limits driven by data
finite_lo  = forest_df["lo_mo"].dropna()
finite_hi  = forest_df["hi_mo"].dropna()
x_min_data = finite_lo.min() if len(finite_lo) else -5
x_max_data = finite_hi.max() if len(finite_hi) else 5
x_pad      = max((x_max_data - x_min_data) * 0.08, 1.0)
ax.set_xlim(x_min_data - x_pad, x_max_data + x_pad)
ax.axvline(x=0, color="#7F8C8D", linewidth=0.9, linestyle="--", zorder=2)

ax.set_yticks(y_positions)
ax.set_yticklabels(forest_df["label"].tolist()[::-1], fontsize=9, ha="right")
ax.set_xlabel(
    f"RMST difference (months, τ={int(TAU)} years)  |  comparison vs reference",
    fontsize=9,
)
ax.set_title(
    f"Adjusted RMST difference by clinical covariate (τ={int(TAU)} years)\n"
    "Each variable adjusted for all others including collection center  |  red = p < 0.05",
    fontsize=9, pad=8,
)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_linewidth(0.6)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="x", labelsize=8, width=0.6, length=3)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", linewidth=0.4, alpha=0.4, zorder=0)

# Right-side annotation using a secondary axis in axes coordinates
# so text positions are independent of data range
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks([])
for sp in ax2.spines.values():
    sp.set_visible(False)

# Column headers (in axes fraction via text transform)
trans = ax.transAxes
ax.text(1.02, 1.01, "n (comp/ref)",
        transform=trans, fontsize=8, fontweight="bold",
        ha="left", va="bottom")
ax.text(1.18, 1.01, "Estimate [95% CI]",
        transform=trans, fontsize=8, fontweight="bold",
        ha="left", va="bottom")
ax.text(1.42, 1.01, "adj p",
        transform=trans, fontsize=8, fontweight="bold",
        ha="left", va="bottom")

# Row values — convert y data coords to axes fraction for text placement
y_lo, y_hi = ax.get_ylim()
for i, (_, row) in enumerate(forest_df.iterrows()):
    y_data  = y_positions[i]
    y_frac  = (y_data - y_lo) / (y_hi - y_lo)
    sig     = (not np.isnan(row["p"])) and row["p"] < 0.05
    col     = "#C0392B" if sig else "#2C3E50"

    n_str = f"{row['n_comp']}/{row['n_ref']}"
    ax.text(1.02, y_frac, n_str,
            transform=trans, fontsize=8,
            ha="left", va="center", color="gray")

    if not np.isnan(row["diff_mo"]):
        est_str = (f"{row['diff_mo']:+.1f} "
                   f"[{row['lo_mo']:+.1f}, {row['hi_mo']:+.1f}]")
        p_str   = f"{row['p']:.3f}" + (" *" if sig else "")
    else:
        est_str = "—"
        p_str   = "—"

    ax.text(1.18, y_frac, est_str,
            transform=trans, fontsize=8,
            ha="left", va="center", color=col)
    ax.text(1.42, y_frac, p_str,
            transform=trans, fontsize=8,
            ha="left", va="center", color=col)

fig.subplots_adjust(right=0.58)
fig.savefig(f"{SAVE_DIR}/forest_rmst_clinical_covariates.png",
            dpi=300, bbox_inches="tight")
fig.savefig(f"{SAVE_DIR}/forest_rmst_clinical_covariates.pdf",
            bbox_inches="tight")
plt.show()
plt.close(fig)

# =============================================================================
# SUMMARY TABLE
# =============================================================================

print("\n" + "="*80)
print(f"RMST SUMMARY (τ={int(TAU)} years, adjusted for all other covariates + center)")
print("="*80)
print(f"{'Covariate':<40} {'n comp/ref':<12} {'Diff (mo)':>10} "
      f"{'95% CI':>22} {'adj p':>8}")
print("-"*80)
for _, row in forest_df.iterrows():
    n_str  = f"{row['n_comp']}/{row['n_ref']}"
    if not np.isnan(row["diff_mo"]):
        ci_str = f"[{row['lo_mo']:+.1f}, {row['hi_mo']:+.1f}]"
        p_str  = f"{row['p']:.3f}" + (" *" if row["p"] < 0.05 else "")
        d_str  = f"{row['diff_mo']:+.1f}"
    else:
        ci_str = "—"
        p_str  = "—"
        d_str  = "—"
    print(f"{row['label']:<40} {n_str:<12} {d_str:>10} {ci_str:>22} {p_str:>8}")

In [ ]:
# FOREST PLOT
# Requires: forest_df already from previous cell, TAU defined


# Aesthetics
FP_FIG_W      = 15       # total figure width (inches)
FP_FIG_H      = 5      # total figure height (inches)
FP_DPI        = 300

FP_FS_HDR     = 8.5      # header font size
FP_FS_ROW     = 8.5      # row text font size
FP_FS_AXIS    = 8.5      # x-axis label font size
FP_FS_TITLE   = 9.5      # title font size

FP_MARKER     = "D"      # marker style
FP_MARKER_SZ  = 5
FP_CAP_SZ     = 3
FP_LW         = 1.3      # error bar line width
FP_ELW        = 1.1      # error bar cap line width

FP_COL_SIG    = "#C0392B"   # color for significant rows
FP_COL_NS     = "#2C3E50"   # color for non-significant rows
FP_COL_N      = "gray"      # color for n column

FP_ROW_H      = 0.55     # row height in inches (controls y spacing)
FP_PLOT_FRAC  = 0.42     # fraction of figure width used by the error bar plot area
                          # remaining fraction is split: labels left, stats right

FP_VLINE_COL  = "#7F8C8D"
FP_GRID_ALPHA = 0.4

FP_SAVE_STEM  = f"{SAVE_DIR}/forest_rmst_clinical_covariates"

# Data (edit labels here)
fp_rows = [
    ("MGMT: methylated vs unmethylated",  res_mgmt,
     res_mgmt["n_arm1"] if res_mgmt else np.nan,
     res_mgmt["n_arm0"] if res_mgmt else np.nan),

    ("Age: >65 vs ≤65",                   res_age,
     res_age["n_arm1"]  if res_age  else np.nan,
     res_age["n_arm0"]  if res_age  else np.nan),

    ("Sex: female vs male",               res_sex,
     res_sex["n_arm1"]  if res_sex  else np.nan,
     res_sex["n_arm0"]  if res_sex  else np.nan),

    ("p53: positive vs negative",         res_p53,
     res_p53["n_arm1"]  if res_p53  else np.nan,
     res_p53["n_arm0"]  if res_p53  else np.nan),

    ("Center: CHUV vs UKER",              res_center.get("CH-ER"),
     res_center["CH-ER"]["n_arm1"] if res_center.get("CH-ER") else np.nan,
     res_center["CH-ER"]["n_arm0"] if res_center.get("CH-ER") else np.nan),

    ("Center: CHUV vs Charité",           res_center.get("CH-CR"),
     res_center["CH-CR"]["n_arm1"] if res_center.get("CH-CR") else np.nan,
     res_center["CH-CR"]["n_arm0"] if res_center.get("CH-CR") else np.nan),

    ("Center: Charité vs UKER",           res_center.get("CR-ER"),
     res_center["CR-ER"]["n_arm1"] if res_center.get("CR-ER") else np.nan,
     res_center["CR-ER"]["n_arm0"] if res_center.get("CR-ER") else np.nan),
]

# xtract values 
def get_res(res):
    if res is None:
        return np.nan, np.nan, np.nan, np.nan
    if "adj_diff" in res:
        return res["adj_diff"]*12, res["adj_lo"]*12, res["adj_hi"]*12, res["adj_p"]
    return res["unadj_diff"]*12, res["unadj_lo"]*12, res["unadj_hi"]*12, res["unadj_p"]

def fmt_p(p):
    if np.isnan(p):
        return "—"
    if p < 0.0001:
        return f"{p:.2e}"
    return f"{p:.4f}"

records = []
for label, res, n1, n0 in fp_rows:
    diff, lo, hi, p = get_res(res)
    records.append({
        "label":   label,
        "n_comp":  int(n1) if not np.isnan(n1) else "—",
        "n_ref":   int(n0) if not np.isnan(n0) else "—",
        "diff_mo": diff,
        "lo_mo":   lo,
        "hi_mo":   hi,
        "p":       p,
        "sig":     (not np.isnan(p)) and p < 0.05,
    })

fp_df = pd.DataFrame(records)
n_rows = len(fp_df)

# Layout
fig, ax = plt.subplots(figsize=(FP_FIG_W, FP_FIG_H))

# y positions: top row = highest index
y_pos = np.arange(n_rows - 1, -1, -1, dtype=float)

# Alternating row shading
for i in range(n_rows):
    if i % 2 == 0:
        ax.axhspan(y_pos[i] - 0.45, y_pos[i] + 0.45,
                   color="whitesmoke", zorder=0)

# Error bars
for i, row in fp_df.iterrows():
    y   = y_pos[i]
    col = FP_COL_SIG if row["sig"] else FP_COL_NS
    if not np.isnan(row["diff_mo"]):
        ax.errorbar(
            x=row["diff_mo"], y=y,
            xerr=[[row["diff_mo"] - row["lo_mo"]],
                  [row["hi_mo"]   - row["diff_mo"]]],
            fmt=FP_MARKER, color=col,
            markersize=FP_MARKER_SZ,
            capsize=FP_CAP_SZ,
            linewidth=FP_LW,
            elinewidth=FP_ELW,
            zorder=3,
        )

# Reference line
ax.axvline(x=0, color=FP_VLINE_COL, linewidth=0.9, linestyle="--", zorder=2)

# x-axis limits
finite_lo  = fp_df["lo_mo"].dropna()
finite_hi  = fp_df["hi_mo"].dropna()
x_min_data = finite_lo.min() if len(finite_lo) else -5
x_max_data = finite_hi.max() if len(finite_hi) else 5
x_pad      = max((x_max_data - x_min_data) * 0.08, 1.0)
ax.set_xlim(x_min_data - x_pad, x_max_data + x_pad)
ax.set_ylim(-0.7, n_rows - 0.3)

# y-axis: no ticks, labels drawn as text to left (see below)
ax.set_yticks([])
ax.set_xlabel(
    f"RMST difference (months, τ={int(TAU)} years)  |  comparison vs reference",
    fontsize=FP_FS_AXIS,
)
ax.set_title(
    f"Adjusted RMST difference by clinical covariate (τ={int(TAU)} years)\n"
    "",
    fontsize=FP_FS_TITLE, pad=8,
)
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_linewidth(0.6)
ax.tick_params(axis="x", labelsize=FP_FS_AXIS, width=0.6, length=3)
ax.grid(axis="x", linewidth=0.4, alpha=FP_GRID_ALPHA, zorder=0)

# Text columns using transAxes
X_LABEL   = -0.52   # covariate label  (right-aligned)
X_N       = -0.02   # n column         (right-aligned)
X_EST     =  1.02   # estimate + CI    (left-aligned)
X_P       =  1.38   # p-value          (left-aligned)

trans = ax.transAxes
y_lo, y_hi = ax.get_ylim()

def y_to_frac(y_data):
    return (y_data - y_lo) / (y_hi - y_lo)

# Headers
header_frac = y_to_frac(n_rows - 0.05)
ax.text(X_LABEL, header_frac, "Covariate",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_N, header_frac, "n (comp/ref)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_EST, header_frac, "Estimate [95% CI] (months)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")
ax.text(X_P, header_frac, "adj p",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")

# Rows
for i, row in fp_df.iterrows():
    yf  = y_to_frac(y_pos[i])
    col = FP_COL_SIG if row["sig"] else FP_COL_NS

    # Covariate label
    ax.text(X_LABEL, yf, row["label"],
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color=FP_COL_NS)

    # n
    n_str = f"{row['n_comp']}/{row['n_ref']}"
    ax.text(X_N, yf, n_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color=FP_COL_N)

    # Estimate [95% CI]
    if not np.isnan(row["diff_mo"]):
        est_str = (f"{row['diff_mo']:+.1f} "
                   f"[{row['lo_mo']:+.1f}, {row['hi_mo']:+.1f}]")
        p_str   = fmt_p(row["p"]) + (" *" if row["sig"] else "")
    else:
        est_str = "—"
        p_str   = "—"

    ax.text(X_EST, yf, est_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)
    ax.text(X_P, yf, p_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)

# Adjust margins
fig.subplots_adjust(left=0.38, right=0.72)

fig.savefig(f"{FP_SAVE_STEM}.png", dpi=FP_DPI, bbox_inches="tight")
fig.savefig(f"{FP_SAVE_STEM}.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved {FP_SAVE_STEM}.png/.pdf")

### Exhuasted T cells in Primary and Recurrent Samples

In [ ]:
# ============================================================================
# CELL 1: T cell phenotypes (terminal exhausted, precursor exhausted, active)
#          in scRNA-seq, matched primary vs recurrent comparison
# ============================================================================

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, mannwhitneyu

sns.set_style("whitegrid")

# ---- Recurrent sample keys --------------------------------------------------
RECURRENT_BLOCKS = [
    "CH_G_164b", "CH_G_168b", "CH_G_180b", "CH_G_186b", "CH_G_189b",
    "CH_G_189c", "CH_G_194b",
    "ER_G_195b", "ER_G_196b", "ER_G_267b", "ER_G_274b", "ER_G_276b",
    "ER_G_278a", "ER_G_282b", "ER_G_284a", "ER_G_288b", "ER_G_292b",
    "ER_G_303b", "ER_G_308b", "ER_G_310b",
]
RECURRENT_KEYS = {b[:9] for b in RECURRENT_BLOCKS}

L1_COL = "celltype_level1_curated"
L2_COL = "celltype_level2_curated"

# ---- T cell signatures ------------------------------------------------------
# Tex_terminal: TCF7-low, multiple inhibitory receptors co-expressed
# Tex_precursor: TCF7+, PD-1+, ICB-responsive
# T_active: cytotoxic effector, can include T effector memory
T_SIGS = {
    "tex_terminal":  ["PDCD1", "HAVCR2", "LAG3", "TIGIT", "TOX", "CTLA4", "ENTPD1"],
    "tex_precursor": ["TCF7", "SLAMF6", "CCR7", "IL7R", "PDCD1"],
    "t_active":      ["GZMB", "GZMA", "GZMK", "PRF1", "IFNG", "NKG7", "GNLY"],
}

# ---- Labels -----------------------------------------------------------------
adata_sc.obs["is_recurrent"] = adata_sc.obs["sample_id"].isin(RECURRENT_KEYS)
adata_sc.obs["patient_root"] = adata_sc.obs["sample_id"].str[:8]
n_rec = adata_sc.obs.loc[adata_sc.obs["is_recurrent"], "sample_id"].nunique()
print(f"scRNA-seq: {adata_sc.obs['is_recurrent'].sum():,} recurrent cells "
      f"from {n_rec} samples")

# ---- T cell scope -----------------------------------------------------------
L1 = adata_sc.obs[L1_COL].astype(str)
L2 = adata_sc.obs[L2_COL].astype(str)

# Use Level 2 T_cell population directly if that's how it's labelled
t_scope_l2 = L2.str.contains("T.?cell|CD8|CD4|Treg", case=False, na=False, regex=True)
t_scope_l1 = L1.str.contains("T.?cell|Lymphoid", case=False, na=False, regex=True)
t_scope = t_scope_l2 if t_scope_l2.any() else t_scope_l1

print(f"T cells in scope: {t_scope.sum():,}")
if t_scope.sum() == 0:
    print("WARNING: T cell scope empty. Check annotation strings:")
    print("L1 values:")
    print(L1.value_counts().head(10))
    print("L2 values:")
    print(L2.value_counts().head(15))

# ---- Score signatures -------------------------------------------------------
_x_backup = adata_sc.X
adata_sc.X = adata_sc.layers["LogNormalize"]
sc_genes = set(adata_sc.var_names)

for name, genes in T_SIGS.items():
    g = [x for x in genes if x in sc_genes]
    if len(g) < 2:
        print(f"  skip {name}: only {len(g)} genes")
        continue
    sc.tl.score_genes(adata_sc, gene_list=g, score_name=f"sc_{name}",
                      ctrl_size=50, n_bins=25, random_state=42)
    print(f"  scored {name} ({len(g)}/{len(genes)} genes)")

adata_sc.X = _x_backup

# ---- Phenotype calls (within T cell scope, mutually exclusive) --------------
t_obs = adata_sc.obs.loc[t_scope]

def q(s, pct):
    return np.nanpercentile(s, pct)

adata_sc.obs["t_phenotype"] = "Other"

mask_term = t_scope & (
    (adata_sc.obs["sc_tex_terminal"] >= q(t_obs["sc_tex_terminal"], 75)) &
    (adata_sc.obs["sc_t_active"]     <  q(t_obs["sc_t_active"], 50))
)
adata_sc.obs.loc[mask_term, "t_phenotype"] = "Tex_terminal"

mask_prec = (
    t_scope
    & (adata_sc.obs["t_phenotype"] == "Other")
    & (adata_sc.obs["sc_tex_precursor"] >= q(t_obs["sc_tex_precursor"], 70))
    & (adata_sc.obs["sc_tex_terminal"]  <  q(t_obs["sc_tex_terminal"], 75))
)
adata_sc.obs.loc[mask_prec, "t_phenotype"] = "Tex_precursor"

mask_act = (
    t_scope
    & (adata_sc.obs["t_phenotype"] == "Other")
    & (adata_sc.obs["sc_t_active"]     >= q(t_obs["sc_t_active"], 70))
    & (adata_sc.obs["sc_tex_terminal"] <  q(t_obs["sc_tex_terminal"], 60))
)
adata_sc.obs.loc[mask_act, "t_phenotype"] = "T_active"

print("\nT cell phenotype distribution:")
print(adata_sc.obs.loc[t_scope, "t_phenotype"].value_counts())

# ---- Per-sample fractions ---------------------------------------------------
per_sample = (
    adata_sc.obs.loc[t_scope]
    .groupby(["sample_id", "is_recurrent", "patient_root"])
    .agg(n_t=("t_phenotype", "size"),
         n_tex_term=("t_phenotype", lambda s: (s == "Tex_terminal").sum()),
         n_tex_prec=("t_phenotype", lambda s: (s == "Tex_precursor").sum()),
         n_t_active=("t_phenotype", lambda s: (s == "T_active").sum()))
    .reset_index()
)
for col in ["n_tex_term", "n_tex_prec", "n_t_active"]:
    per_sample[col.replace("n_", "frac_")] = (
        per_sample[col] / per_sample["n_t"].replace(0, np.nan)
    )
per_sample = per_sample[per_sample["n_t"] >= 10].copy()
print(f"\nSamples with ≥10 T cells: {len(per_sample)} "
      f"(primary={(~per_sample['is_recurrent']).sum()}, "
      f"recurrent={per_sample['is_recurrent'].sum()})")

# ---- Paired test on matched primary-recurrent patients ----------------------
print("\n" + "=" * 70)
print("MATCHED PRIMARY vs RECURRENT (paired Wilcoxon)")
print("=" * 70)

paired_results = []
for frac_col, pheno in [("frac_tex_term", "Tex_terminal"),
                         ("frac_tex_prec", "Tex_precursor"),
                         ("frac_t_active", "T_active")]:
    paired = per_sample.groupby("patient_root").filter(
        lambda g: g["is_recurrent"].nunique() == 2
    )
    pivot = paired.pivot_table(
        index="patient_root", columns="is_recurrent",
        values=frac_col, aggfunc="mean"
    ).dropna()
    if len(pivot) < 3:
        print(f"  {pheno:14s}: only {len(pivot)} matched pairs, skipping")
        continue
    delta = pivot[True] - pivot[False]
    w, p = wilcoxon(delta, alternative="two-sided")
    direction = "↑ recurrent" if delta.median() > 0 else "↓ recurrent"
    print(f"  {pheno:14s}  n_pairs={len(pivot):2d}  "
          f"primary={pivot[False].median():.4f}  "
          f"recurrent={pivot[True].median():.4f}  "
          f"Δ={delta.median():+.4f}  {direction}  p={p:.3g}")
    paired_results.append({
        "phenotype": pheno, "n_pairs": len(pivot),
        "median_primary": pivot[False].median(),
        "median_recurrent": pivot[True].median(),
        "median_delta": delta.median(), "wilcoxon_p": p,
    })

# ---- Unpaired test on all samples -------------------------------------------
print("\n" + "=" * 70)
print("ALL SAMPLES (Mann-Whitney)")
print("=" * 70)
for frac_col, pheno in [("frac_tex_term", "Tex_terminal"),
                         ("frac_tex_prec", "Tex_precursor"),
                         ("frac_t_active", "T_active")]:
    prim = per_sample.loc[~per_sample["is_recurrent"], frac_col].dropna()
    rec = per_sample.loc[per_sample["is_recurrent"], frac_col].dropna()
    if len(prim) >= 3 and len(rec) >= 3:
        u, p = mannwhitneyu(rec, prim, alternative="two-sided")
        print(f"  {pheno:14s}  primary n={len(prim):3d} median={prim.median():.4f}  "
              f"recurrent n={len(rec):2d} median={rec.median():.4f}  p={p:.3g}")

# ---- Figure -----------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top-left: unpaired comparison across phenotypes
ax = axes[0, 0]
plot_df = per_sample.melt(
    id_vars=["sample_id", "is_recurrent", "patient_root"],
    value_vars=["frac_tex_term", "frac_tex_prec", "frac_t_active"],
    var_name="phenotype", value_name="fraction"
)
plot_df["phenotype"] = plot_df["phenotype"].str.replace("frac_", "")
plot_df["group"] = np.where(plot_df["is_recurrent"], "Recurrent", "Primary")
sns.boxplot(data=plot_df, x="phenotype", y="fraction", hue="group", ax=ax,
            order=["tex_term", "tex_prec", "t_active"],
            hue_order=["Primary", "Recurrent"], showfliers=False)
ax.set_title("All samples: T cell phenotype fractions")
ax.set_xlabel(""); ax.set_ylabel("Fraction of T cells")
ax.legend(fontsize=8)

# Top-middle through bottom-right: paired plots
phenos = [("Tex_terminal", "frac_tex_term"),
          ("Tex_precursor", "frac_tex_prec"),
          ("T_active", "frac_t_active")]

for i, (pheno, frac_col) in enumerate(phenos):
    ax = axes[(i + 1) // 3, (i + 1) % 3] if i < 2 else axes[1, 0]
    if i == 0:
        ax = axes[0, 1]
    elif i == 1:
        ax = axes[0, 2]
    else:
        ax = axes[1, 0]

    paired = per_sample.groupby("patient_root").filter(
        lambda g: g["is_recurrent"].nunique() == 2
    )
    pivot = paired.pivot_table(
        index="patient_root", columns="is_recurrent",
        values=frac_col, aggfunc="mean"
    ).dropna()

    if len(pivot) == 0:
        ax.text(0.5, 0.5, f"No matched pairs",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(pheno)
        continue

    for patient in pivot.index:
        ax.plot([0, 1], [pivot.loc[patient, False], pivot.loc[patient, True]],
                "o-", color="gray", alpha=0.6, markersize=5)
    # Median line in color
    ax.plot([0, 1], [pivot[False].median(), pivot[True].median()],
            "o-", color="#C0392B", linewidth=2.5, markersize=8,
            label="Median", zorder=10)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Primary", "Recurrent"])
    ax.set_ylabel("Fraction")
    if len(pivot) >= 3:
        delta = pivot[True] - pivot[False]
        w, p = wilcoxon(delta, alternative="two-sided")
        ax.set_title(f"{pheno}\nn={len(pivot)} pairs, paired Wilcoxon p={p:.3g}",
                     fontsize=10)
    else:
        ax.set_title(f"{pheno} (n={len(pivot)} pairs)")
    ax.legend(fontsize=8)

# Bottom middle and right: score distributions for context
for i, (col, title) in enumerate([
    ("sc_tex_terminal", "Terminal exhaustion score"),
    ("sc_t_active", "Active cytotoxic score"),
]):
    ax = axes[1, 1 + i]
    plot_df = adata_sc.obs.loc[t_scope, [col, "is_recurrent"]].copy()
    plot_df["group"] = np.where(plot_df["is_recurrent"], "Recurrent", "Primary")
    sns.violinplot(data=plot_df, x="group", y=col, ax=ax,
                   order=["Primary", "Recurrent"], cut=0)
    ax.set_title(title)
    ax.set_xlabel("")

plt.tight_layout()
plt.savefig("tcell_phenotypes_primary_vs_recurrent.png", dpi=200, bbox_inches="tight")
plt.show()